## 1️⃣ Environment Setup & GPU Verification

In [36]:
# ============================================================
# 1.1 Verify GPU is available
# ============================================================
import subprocess

print("=" * 70)
print("🔍 Checking GPU availability...")
print("=" * 70)

# Check NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print("❌ ERROR: No NVIDIA GPU detected!")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
    raise RuntimeError("GPU not available")

print("\n✅ GPU detected! Proceeding with setup...")

🔍 Checking GPU availability...
Thu Jan  8 00:29:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             62W /  400W |     707MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------

In [37]:
# ============================================================
# 1.2 Install dependencies (Colab with TF 2.19.0)
# ============================================================
# ⚠️ If you see numpy/h5py errors, go to Runtime > Restart runtime
# then run cells 1 and 2 again (skip pip install, just verify)

print("📦 Checking/Installing dependencies...")

# Check if we need to install anything
import subprocess
result = subprocess.run(['pip', 'show', 'xgboost'], capture_output=True, text=True)
if 'not found' in result.stderr.lower() or result.returncode != 0:
    print("   Installing additional packages...")
    !pip install -q xgboost>=2.0.3 rich>=13.7.1 python-dotenv>=1.0.0 structlog>=24.1.0
else:
    print("   ✓ Packages already installed")

# Verify key packages
import tensorflow as tf
import numpy as np
import pandas as pd

print(f"\n✅ Dependencies ready!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   GPU: {tf.config.list_physical_devices('GPU')}")

📦 Checking/Installing dependencies...
   ✓ Packages already installed

✅ Dependencies ready!
   TensorFlow: 2.19.0
   NumPy: 2.0.2
   Pandas: 2.2.2
   GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [38]:
# ============================================================
# 1.3 Verify TensorFlow CUDA setup & Optimize for A100
# ============================================================
import tensorflow as tf

print("=" * 70)
print("🔧 TensorFlow Configuration")
print("=" * 70)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

# Enable memory growth to prevent OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✅ Memory growth enabled for {len(gpus)} GPU(s)")
        
        # Get GPU details
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        gpu_name = gpu_details.get('device_name', 'Unknown')
        print(f"   GPU: {gpu_name}")
        
        # A100-specific optimizations
        if 'A100' in gpu_name:
            print(f"\n🚀 A100 POWERHOUSE MODE ENABLED!")
            print(f"   • 80GB VRAM → 4x larger model (d_model=128, 4 layers)")
            print(f"   • batch_size=256 (4x Mac M1)")
            print(f"   • seq_len=128 (2x temporal context)")
            print(f"   • TF32 enabled for Tensor Core acceleration")
            # Enable TF32 for A100 (faster than FP32, same accuracy)
            tf.config.experimental.enable_tensor_float_32_execution(True)
    except RuntimeError as e:
        print(f"⚠️ Could not set memory growth: {e}")

# ⚠️ MIXED PRECISION DISABLED - causes 0% accuracy bug in TF 2.19
# TF32 alone provides ~1.5x speedup without this issue
print(f"\n✅ Using float32 precision (TF32 enabled for A100)")
print("   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug")

🔧 TensorFlow Configuration
TensorFlow version: 2.19.0
CUDA available: True
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ Memory growth enabled for 1 GPU(s)
   GPU: NVIDIA A100-SXM4-80GB

🚀 A100 POWERHOUSE MODE ENABLED!
   • 80GB VRAM → 4x larger model (d_model=128, 4 layers)
   • batch_size=256 (4x Mac M1)
   • seq_len=128 (2x temporal context)
   • TF32 enabled for Tensor Core acceleration

✅ Using float32 precision (TF32 enabled for A100)
   Note: Mixed precision disabled due to TF 2.19 accuracy metric bug


In [39]:
# ============================================================
# 1.4 REGISTER CUSTOM LOSS FUNCTIONS (REQUIRED FOR MODEL LOADING)
# ============================================================
# These MUST be registered BEFORE loading any saved models
# Run this cell early in the notebook to avoid "Unknown loss function" errors

import tensorflow as tf
import keras.backend as K

# ===== DEFINE CUSTOM LOSS FUNCTIONS =====

def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """Focal Loss for handling class imbalance"""
    y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1 - K.epsilon())
    pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
    alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
    return -tf.reduce_mean(alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt))

def anti_collapse_loss(y_true, y_pred, lambda_entropy=0.1):
    """Anti-collapse loss that penalizes low entropy predictions"""
    y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1 - K.epsilon())
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    entropy = -tf.reduce_mean(y_pred * tf.math.log(y_pred) + 
                              (1 - y_pred) * tf.math.log(1 - y_pred))
    return bce - lambda_entropy * entropy

def combined_loss(y_true, y_pred):
    """Combined focal + anti-collapse loss - MAIN TRAINING LOSS"""
    focal = focal_loss(y_true, y_pred)
    anti_collapse = anti_collapse_loss(y_true, y_pred)
    return 0.7 * focal + 0.3 * anti_collapse

# ===== REGISTER WITH KERAS CUSTOM OBJECTS (VERSION-COMPATIBLE WAY) =====

# Register all custom losses in Keras custom objects registry
tf.keras.utils.get_custom_objects().update({
    'focal_loss': focal_loss,
    'anti_collapse_loss': anti_collapse_loss,
    'combined_loss': combined_loss
})

# Create custom_objects dict for explicit load_model calls
CUSTOM_OBJECTS = {
    'focal_loss': focal_loss,
    'anti_collapse_loss': anti_collapse_loss,
    'combined_loss': combined_loss
}

print("=" * 70)
print("✅ CUSTOM LOSS FUNCTIONS REGISTERED GLOBALLY")
print("=" * 70)
print("Registered losses:")
print("  • focal_loss - Handles class imbalance")
print("  • anti_collapse_loss - Prevents prediction collapse")
print("  • combined_loss - Main training loss (70% focal + 30% anti-collapse)")
print("\n📌 These are now available for ALL model loading operations")
print("📌 Use CUSTOM_OBJECTS dict for explicit load_model calls")
print("\n✅ Compatible with TensorFlow 2.x and Keras 3.x")

✅ CUSTOM LOSS FUNCTIONS REGISTERED GLOBALLY
Registered losses:
  • focal_loss - Handles class imbalance
  • anti_collapse_loss - Prevents prediction collapse
  • combined_loss - Main training loss (70% focal + 30% anti-collapse)

📌 These are now available for ALL model loading operations
📌 Use CUSTOM_OBJECTS dict for explicit load_model calls

✅ Compatible with TensorFlow 2.x and Keras 3.x


## 2️⃣ Clone Repository & Setup

In [40]:
# ============================================================
# 2.1 Clone the ML Engine repository
# ============================================================
# ⚠️ RE-RUN THIS CELL if you see dimension mismatch errors!
# This pulls the latest code with bug fixes from GitHub.

import os

REPO_URL = "https://github.com/Raynergy-svg/ml_engine.git"
REPO_DIR = "/content/ml_engine"

# IMPORTANT: Reset to /content first (fixes "getcwd" errors after rm -rf)
os.chdir("/content")

# Remove existing directory if it exists (forces fresh clone)
if os.path.exists(REPO_DIR):
    print("🗑️ Removing existing repo to get latest fixes...")
    !rm -rf {REPO_DIR}

print(f"📥 Cloning repository from {REPO_URL}...")
!git clone {REPO_URL} {REPO_DIR}

# Change to repo directory
os.chdir(REPO_DIR)
print(f"\n📂 Working directory: {os.getcwd()}")

# Show latest commit to verify we have the fix
print("\n📋 Latest commit:")
!git log --oneline -3

print("\n📁 Repository contents:")
!ls -la

🗑️ Removing existing repo to get latest fixes...
📥 Cloning repository from https://github.com/Raynergy-svg/ml_engine.git...
Cloning into '/content/ml_engine'...
remote: Enumerating objects: 1786, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 1786 (delta 136), reused 155 (delta 89), pack-reused 1570 (from 3)
Receiving objects: 100% (1786/1786), 381.29 MiB | 16.10 MiB/s, done.
Resolving deltas: 100% (900/900), done.
Updating files: 100% (567/567), done.

📂 Working directory: /content/ml_engine

📋 Latest commit:
2a79b5c (HEAD -> main, origin/main, origin/HEAD) Add initial training data and configuration files for trading model
20ed22a Enhance training process by adding support for multiple models: Transformer, XGBoost, Random Forest, Ridge, and HistGradientBoosting. Implement detailed logging for each training step and ensure proper data handling for direction data. Introduce warm-start functionality for Transformer 

In [41]:
# ============================================================
# 2.2 Create necessary directories & Clear stale replay buffers
# ============================================================
import os
import glob
from pathlib import Path

directories = [
    "trained_data/models",
    "trained_data/checkpoints",
    "trained_data/checkpoints/tensorflow",
    "trained_data/replay/EUR_USD",
    "trained_data/replay/USD_JPY",
    "trained_data/replay/GBP_USD",
    "trained_data/logs",
    "trained_data/scalers",
    "market_data",
]

for d in directories:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {d}")

# Clear any stale replay buffers (they may have different feature dimensions)
replay_files = glob.glob("trained_data/replay/**/*.npz", recursive=True)
if replay_files:
    print(f"\n🗑️ Clearing {len(replay_files)} stale replay buffer(s)...")
    for f in replay_files:
        os.remove(f)
        print(f"   Removed: {f}")
    print("✅ Replay buffers cleared (prevents feature dimension mismatch)")
else:
    print("\n✅ No stale replay buffers to clear")

print("\n📁 Directory structure ready!")

✅ Created: trained_data/models
✅ Created: trained_data/checkpoints
✅ Created: trained_data/checkpoints/tensorflow
✅ Created: trained_data/replay/EUR_USD
✅ Created: trained_data/replay/USD_JPY
✅ Created: trained_data/replay/GBP_USD
✅ Created: trained_data/logs
✅ Created: trained_data/scalers
✅ Created: market_data

🗑️ Clearing 2 stale replay buffer(s)...
   Removed: trained_data/replay/USD_JPY/buffer.npz
   Removed: trained_data/replay/EUR_USD/buffer.npz
✅ Replay buffers cleared (prevents feature dimension mismatch)

📁 Directory structure ready!


## 3️⃣ Environment Variables (OANDA API)

In [42]:
# ============================================================
# 3.1 Set OANDA API credentials
# ============================================================
import os
from getpass import getpass

print("=" * 70)
print("🔐 OANDA API Configuration")
print("=" * 70)
print("\nEnter your OANDA Practice account credentials.")
print("(These are stored only in this session's memory)\n")

# Interactive input with clear format hints
print("API Token format: xxxx-xxxx (long string with hyphen)")
OANDA_API_TOKEN = getpass("OANDA API Token: ")

print("\nAccount ID format: 101-001-XXXXXXXX-001")
OANDA_ACCOUNT_ID = input("OANDA Account ID: ")

# Validate inputs
if '-' in OANDA_ACCOUNT_ID and len(OANDA_ACCOUNT_ID) > 50:
    print("\n⚠️ WARNING: Account ID looks like a token - you may have swapped them!")
    print("   Swapping automatically...")
    OANDA_API_TOKEN, OANDA_ACCOUNT_ID = OANDA_ACCOUNT_ID, OANDA_API_TOKEN

# Set environment variables
os.environ["OANDA_API_TOKEN"] = OANDA_API_TOKEN
os.environ["OANDA_ACCOUNT_ID"] = OANDA_ACCOUNT_ID

# Verify (show only last 4 chars of token)
print(f"\n✅ OANDA_API_TOKEN: ...{OANDA_API_TOKEN[-4:]}")
print(f"✅ OANDA_ACCOUNT_ID: {OANDA_ACCOUNT_ID}")

🔐 OANDA API Configuration

Enter your OANDA Practice account credentials.
(These are stored only in this session's memory)

API Token format: xxxx-xxxx (long string with hyphen)

Account ID format: 101-001-XXXXXXXX-001

✅ OANDA_API_TOKEN: ...1a62
✅ OANDA_ACCOUNT_ID: 101-001-37949116-001


In [60]:
# ============================================================
# 3.2 Test OANDA connection
# ============================================================
import sys
sys.path.insert(0, '/content/ml_engine')

try:
    from oanda_practice import OandaPracticeClient
    
    client = OandaPracticeClient.from_env()
    print("✅ OANDA client initialized successfully!")
    print("\n📊 Testing candle fetch...")
    
    # Fetch a small sample to verify connection
    resp = client.get_candles(
        instrument="EUR_USD",
        granularity="H1",
        count=10
    )
    # Response is a dict with "candles" key
    candles = resp.get("candles", []) if isinstance(resp, dict) else []
    
    print(f"✅ Fetched {len(candles)} candles from OANDA")
    if candles:
        c = candles[-1]
        print(f"   Latest: {c['time']} Close: {c['mid']['c']}")
    
except Exception as e:
    print(f"❌ OANDA connection failed: {e}")
    import traceback
    traceback.print_exc()
    print("\n⚠️ You can still train using local CSV files.")
    print("   Upload your market data to /content/ml_engine/market_data/")

✅ OANDA client initialized successfully!

📊 Testing candle fetch...
✅ Fetched 10 candles from OANDA
   Latest: 2026-01-08T00:00:00.000000000Z Close: 1.16767


## 4️⃣ Data Preparation

In [61]:
# ============================================================
# 4.1 Configure Data Size for Training
# ============================================================
# ⚠️ ADJUST THIS TO CONTROL TRAINING TIME!
# More data = better model but slower fetching

import numpy as np

# === USER CONFIGURATION ===
# Choose your data size preset:
#   "quick"  = 1 year  (~6,000 candles/pair)  - Fast testing, ~5 min fetch
#   "normal" = 2 years (~12,000 candles/pair) - Good balance, ~10 min fetch  
#   "full"   = 3 years (~18,000 candles/pair) - Better model, ~15 min fetch
#   "max"    = 5 years (~30,000 candles/pair) - Maximum practical, ~25 min fetch

DATA_SIZE_PRESET = "normal"  # ⬅️ CHANGE THIS

# Or set exact candles per pair (overrides preset if > 0)
CUSTOM_CANDLES_PER_PAIR = 0  # Set to e.g. 15000 for custom amount

# === PRESET DEFINITIONS ===
DATA_PRESETS = {
    "quick":  6000,   # ~1 year H1
    "normal": 12000,  # ~2 years H1
    "full":   18000,  # ~3 years H1
    "max":    30000,  # ~5 years H1
}

# Define A100 config (used by training cells)
A100_CONFIG = {
    "transformer_d_model": 128,
    "transformer_num_heads": 8,
    "transformer_num_layers": 4,
    "transformer_dff": 512,
    "seq_len": 128,
    "batch_size": 256,
}

# Calculate model params for reference
d_model = A100_CONFIG["transformer_d_model"]
num_layers = A100_CONFIG["transformer_num_layers"]
dff = A100_CONFIG["transformer_dff"]
n_features = 80
embed_params = n_features * d_model
attn_params = 4 * d_model * d_model
ffn_params = 2 * d_model * dff
layer_params = attn_params + ffn_params + 4 * d_model
total_params = embed_params + (num_layers * layer_params) + d_model

# Determine candles per pair
if CUSTOM_CANDLES_PER_PAIR > 0:
    CANDLES_PER_PAIR = CUSTOM_CANDLES_PER_PAIR
    preset_name = "custom"
else:
    CANDLES_PER_PAIR = DATA_PRESETS.get(DATA_SIZE_PRESET, DATA_PRESETS["normal"])
    preset_name = DATA_SIZE_PRESET

# Calculate estimates
years_of_data = CANDLES_PER_PAIR / 6240  # 6240 = H1 candles per year
fetch_time_min = CANDLES_PER_PAIR / 5000 * 0.5  # ~0.5 min per 5000 candles

print("=" * 70)
print("📊 DATA SIZE CONFIGURATION")
print("=" * 70)
print(f"\n🎚️ Preset: {preset_name.upper()}")
print(f"   CANDLES_PER_PAIR = {CANDLES_PER_PAIR:,}")
print(f"   ≈ {years_of_data:.1f} years of H1 data per pair")
print(f"\n🔧 Model: ~{total_params:,} parameters")
print(f"\n⏱️ Estimated fetch time per pair: ~{fetch_time_min:.1f} minutes")
print(f"\n📊 Available presets:")
for name, candles in DATA_PRESETS.items():
    marker = " ⬅️ SELECTED" if name == preset_name else ""
    print(f"   • {name:8} = {candles:,} candles (~{candles/6240:.1f} years){marker}")

# Set OPTIMAL_CANDLES for backward compatibility with other cells
OPTIMAL_CANDLES = CANDLES_PER_PAIR
print(f"\n✅ Ready to fetch {CANDLES_PER_PAIR:,} candles per pair")

📊 DATA SIZE CONFIGURATION

🎚️ Preset: NORMAL
   CANDLES_PER_PAIR = 12,000
   ≈ 1.9 years of H1 data per pair

🔧 Model: ~798,848 parameters

⏱️ Estimated fetch time per pair: ~1.2 minutes

📊 Available presets:
   • quick    = 6,000 candles (~1.0 years)
   • normal   = 12,000 candles (~1.9 years) ⬅️ SELECTED
   • full     = 18,000 candles (~2.9 years)
   • max      = 30,000 candles (~4.8 years)

✅ Ready to fetch 12,000 candles per pair


In [62]:
# ============================================================
# 4.2 Multi-Pair Scanner & Data Fetcher
# ============================================================
# Scans top pairs and fetches optimal data for A100 training

import asyncio
import pandas as pd
import numpy as np
from typing import List, Dict
from datetime import datetime, timedelta

# === CONFIGURATION ===
MULTI_PAIR_MODE = True  # Set to False for single pair training

# Top FX pairs by liquidity (buddy scan candidates)
SCAN_PAIRS = [
    "EUR_USD",  # Most liquid
    "USD_JPY",  # Second most liquid  
    "GBP_USD",  # Cable
    "USD_CHF",  # Swissy
    "AUD_USD",  # Aussie
    "USD_CAD",  # Loonie
    "NZD_USD",  # Kiwi
    "EUR_GBP",  # Cross
    "EUR_JPY",  # Cross
    "GBP_JPY",  # Cross (volatile)
]

GRANULARITY = "H1"  # H1 for 24-bar daily lookahead

def fetch_pair_data(client, instrument: str, target_candles: int) -> pd.DataFrame:
    """Fetch candles for a single pair with progress tracking."""
    all_candles = []
    from_time = None
    
    print(f"\n   📊 {instrument}: Fetching {target_candles:,} candles...")
    
    while len(all_candles) < target_candles:
        remaining = target_candles - len(all_candles)
        batch_size = min(5000, remaining)
        
        try:
            if from_time:
                resp = client.get_candles(
                    instrument=instrument,
                    granularity=GRANULARITY,
                    count=batch_size,
                    to_time=from_time
                )
            else:
                resp = client.get_candles(
                    instrument=instrument,
                    granularity=GRANULARITY,
                    count=batch_size
                )
            
            candles = resp.get("candles", []) if isinstance(resp, dict) else []
            
            if not candles:
                print(f"      ⚠️ No more data available ({len(all_candles):,} fetched)")
                break
                
            all_candles = candles + all_candles
            from_time = candles[0]['time']
            
            # Progress
            pct = min(100, len(all_candles) / target_candles * 100)
            print(f"      Progress: {len(all_candles):,}/{target_candles:,} ({pct:.0f}%)", end='\r')
            
        except Exception as e:
            print(f"      ❌ Error: {e}")
            break
    
    if not all_candles:
        return None
    
    # Convert to DataFrame
    df = pd.DataFrame([{
        'time': c['time'],
        'open': float(c['mid']['o']),
        'high': float(c['mid']['h']),
        'low': float(c['mid']['l']),
        'close': float(c['mid']['c']),
        'volume': int(c['volume']),
        'instrument': instrument
    } for c in all_candles])
    
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.sort_index(inplace=True)
    
    print(f"      ✅ Got {len(df):,} candles ({df.index[0].date()} to {df.index[-1].date()})")
    
    return df

def buddy_scan_pairs(pairs: List[str], client) -> List[Dict]:
    """
    Scan pairs for trading suitability (simplified buddy scan).
    Returns pairs ranked by volatility and trend strength.
    """
    print("=" * 70)
    print("🔍 BUDDY SCAN: Analyzing pairs for training suitability...")
    print("=" * 70)
    
    results = []
    
    for pair in pairs:
        try:
            # Fetch recent data for analysis
            resp = client.get_candles(
                instrument=pair,
                granularity="H1",
                count=500  # ~3 weeks for quick scan
            )
            candles = resp.get("candles", []) if isinstance(resp, dict) else []
            
            if len(candles) < 100:
                continue
            
            # Calculate metrics
            closes = np.array([float(c['mid']['c']) for c in candles])
            highs = np.array([float(c['mid']['h']) for c in candles])
            lows = np.array([float(c['mid']['l']) for c in candles])
            
            # Volatility (ATR-like)
            ranges = highs - lows
            avg_range = np.mean(ranges)
            volatility = avg_range / np.mean(closes) * 100  # As percentage
            
            # Trend strength (price change / volatility)
            price_change = abs(closes[-1] - closes[0]) / closes[0] * 100
            trend_strength = price_change / (volatility * np.sqrt(len(candles)/24))
            
            # Directional moves (% of bars with clear direction)
            returns = np.diff(closes) / closes[:-1]
            clear_moves = np.sum(np.abs(returns) > 0.0005) / len(returns) * 100
            
            # Score: balance of volatility (tradeable) and trend (predictable)
            score = volatility * 0.3 + trend_strength * 0.4 + clear_moves * 0.003
            
            results.append({
                'pair': pair,
                'volatility': volatility,
                'trend_strength': trend_strength,
                'clear_moves_pct': clear_moves,
                'score': score,
                'current_price': closes[-1]
            })
            
            print(f"   {pair}: vol={volatility:.3f}%, trend={trend_strength:.2f}, score={score:.3f}")
            
        except Exception as e:
            print(f"   {pair}: ❌ Error - {e}")
    
    # Sort by score
    results.sort(key=lambda x: x['score'], reverse=True)
    
    print(f"\n🏆 Top pairs for training:")
    for i, r in enumerate(results[:5], 1):
        print(f"   {i}. {r['pair']} (score: {r['score']:.3f})")
    
    return results

# === RUN BUDDY SCAN ===
if MULTI_PAIR_MODE:
    scan_results = buddy_scan_pairs(SCAN_PAIRS, client)
    
    # Select top N pairs for training
    TOP_N_PAIRS = 5  # Train on top 5 pairs
    SELECTED_PAIRS = [r['pair'] for r in scan_results[:TOP_N_PAIRS]]
    
    print(f"\n✅ Selected {len(SELECTED_PAIRS)} pairs for multi-pair training:")
    print(f"   {', '.join(SELECTED_PAIRS)}")
else:
    SELECTED_PAIRS = ["EUR_USD"]
    print(f"ℹ️ Single-pair mode: {SELECTED_PAIRS[0]}")

🔍 BUDDY SCAN: Analyzing pairs for training suitability...
   EUR_USD: vol=0.089%, trend=0.43, score=0.295
   USD_JPY: vol=0.116%, trend=1.93, score=0.931
   GBP_USD: vol=0.104%, trend=1.91, score=0.911
   USD_CHF: vol=0.110%, trend=1.45, score=0.733
   AUD_USD: vol=0.125%, trend=1.95, score=0.948
   USD_CAD: vol=0.075%, trend=0.83, score=0.430
   NZD_USD: vol=0.138%, trend=0.46, score=0.369
   EUR_GBP: vol=0.070%, trend=2.26, score=0.992
   EUR_JPY: vol=0.103%, trend=2.56, score=1.169
   GBP_JPY: vol=0.112%, trend=3.79, score=1.682

🏆 Top pairs for training:
   1. GBP_JPY (score: 1.682)
   2. EUR_JPY (score: 1.169)
   3. EUR_GBP (score: 0.992)
   4. AUD_USD (score: 0.948)
   5. USD_JPY (score: 0.931)

✅ Selected 5 pairs for multi-pair training:
   GBP_JPY, EUR_JPY, EUR_GBP, AUD_USD, USD_JPY


In [63]:
# ============================================================
# 4.3 Fetch Data for All Selected Pairs
# ============================================================
import os

# Use CANDLES_PER_PAIR from cell 4.1 (preset-based)
print(f"\n📊 Fetching {CANDLES_PER_PAIR:,} candles per pair ({len(SELECTED_PAIRS)} pairs)")
print(f"   Total data: ~{CANDLES_PER_PAIR * len(SELECTED_PAIRS):,} candles")
print(f"   Estimated time: ~{len(SELECTED_PAIRS) * CANDLES_PER_PAIR / 5000 * 0.5:.0f} minutes")

# Fetch data for each pair
pair_dataframes = {}
DATA_PATHS = {}

print("\n" + "=" * 70)
print("📥 FETCHING DATA")
print("=" * 70)

for i, pair in enumerate(SELECTED_PAIRS, 1):
    print(f"\n[{i}/{len(SELECTED_PAIRS)}] {pair}")
    df = fetch_pair_data(client, pair, CANDLES_PER_PAIR)
    
    if df is not None and len(df) > 1000:  # Minimum viable data
        pair_dataframes[pair] = df
        
        # Save to disk
        safe_name = pair.replace('_', '')
        data_path = f"/content/ml_engine/market_data/{safe_name}_{GRANULARITY}.csv"
        df.to_csv(data_path)
        DATA_PATHS[pair] = data_path
        
        print(f"   💾 Saved: {data_path}")
    else:
        print(f"   ⚠️ Skipping {pair} - insufficient data")

print(f"\n" + "=" * 70)
print(f"✅ Fetched data for {len(pair_dataframes)} pairs:")
for pair, df in pair_dataframes.items():
    date_range = f"{df.index[0].date()} to {df.index[-1].date()}"
    print(f"   • {pair}: {len(df):,} candles ({date_range})")

# Total candles
TOTAL_CANDLES = sum(len(df) for df in pair_dataframes.values())
print(f"\n📊 Total training data: {TOTAL_CANDLES:,} candles")


📊 Fetching 12,000 candles per pair (5 pairs)
   Total data: ~60,000 candles
   Estimated time: ~6 minutes

📥 FETCHING DATA

[1/5] GBP_JPY

   📊 GBP_JPY: Fetching 12,000 candles...
      ✅ Got 12,000 candles (2024-02-01 to 2026-01-08)
   💾 Saved: /content/ml_engine/market_data/GBPJPY_H1.csv

[2/5] EUR_JPY

   📊 EUR_JPY: Fetching 12,000 candles...
      ✅ Got 12,000 candles (2024-02-01 to 2026-01-08)
   💾 Saved: /content/ml_engine/market_data/EURJPY_H1.csv

[3/5] EUR_GBP

   📊 EUR_GBP: Fetching 12,000 candles...
      ✅ Got 12,000 candles (2024-02-01 to 2026-01-08)
   💾 Saved: /content/ml_engine/market_data/EURGBP_H1.csv

[4/5] AUD_USD

   📊 AUD_USD: Fetching 12,000 candles...
      ✅ Got 12,000 candles (2024-02-01 to 2026-01-08)
   💾 Saved: /content/ml_engine/market_data/AUDUSD_H1.csv

[5/5] USD_JPY

   📊 USD_JPY: Fetching 12,000 candles...
      ✅ Got 12,000 candles (2024-02-01 to 2026-01-08)
   💾 Saved: /content/ml_engine/market_data/USDJPY_H1.csv

✅ Fetched data for 5 pairs:
   • GB

In [64]:
# ============================================================
# 4.4 Combine Multi-Pair Data for Training
# ============================================================
# Combines all pairs into unified training dataset with pair embeddings

def prepare_multi_pair_data(pair_dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Combine multiple pair DataFrames into unified training dataset.
    Adds pair identifier for potential pair-specific learning.
    """
    combined_dfs = []
    
    for pair, df in pair_dfs.items():
        df = df.copy()
        
        # Add pair identifier (for potential embedding)
        df['pair'] = pair
        df['pair_id'] = list(pair_dfs.keys()).index(pair)
        
        # Normalize OHLCV relative to close (makes pairs comparable)
        df['norm_open'] = df['open'] / df['close']
        df['norm_high'] = df['high'] / df['close']
        df['norm_low'] = df['low'] / df['close']
        df['norm_close'] = 1.0  # Always 1 after normalization
        
        # Returns (universal across pairs)
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
        
        combined_dfs.append(df)
    
    # Concatenate all pairs
    combined = pd.concat(combined_dfs, axis=0)
    
    # Sort by time (interleaves pairs)
    combined = combined.sort_index()
    
    # Remove NaN from returns calculation
    combined = combined.dropna()
    
    return combined

if MULTI_PAIR_MODE and len(pair_dataframes) > 1:
    print("=" * 70)
    print("🔀 COMBINING MULTI-PAIR DATA")
    print("=" * 70)
    
    combined_df = prepare_multi_pair_data(pair_dataframes)
    
    print(f"\n📊 Combined dataset statistics:")
    print(f"   Total samples: {len(combined_df):,}")
    print(f"   Date range: {combined_df.index.min()} to {combined_df.index.max()}")
    print(f"\n   Samples per pair:")
    for pair in SELECTED_PAIRS:
        pair_count = len(combined_df[combined_df['pair'] == pair])
        print(f"      {pair}: {pair_count:,}")
    
    # Save combined data
    COMBINED_DATA_PATH = "/content/ml_engine/market_data/MULTI_PAIR_H1.csv"
    combined_df.to_csv(COMBINED_DATA_PATH)
    print(f"\n   💾 Saved: {COMBINED_DATA_PATH}")
    
    # Use combined for training
    DATA_PATH = COMBINED_DATA_PATH
    CANDLES = len(combined_df)
    INSTRUMENT = "MULTI_PAIR"
    
    print(f"\n✅ Multi-pair data ready: {CANDLES:,} samples")
else:
    # Single pair mode - use first pair
    INSTRUMENT = SELECTED_PAIRS[0]
    DATA_PATH = DATA_PATHS.get(INSTRUMENT)
    CANDLES = len(pair_dataframes.get(INSTRUMENT, pd.DataFrame()))
    print(f"ℹ️ Single-pair mode: {INSTRUMENT} ({CANDLES:,} candles)")

🔀 COMBINING MULTI-PAIR DATA

📊 Combined dataset statistics:
   Total samples: 59,995
   Date range: 2024-02-02 00:00:00+00:00 to 2026-01-08 00:00:00+00:00

   Samples per pair:
      GBP_JPY: 11,999
      EUR_JPY: 11,999
      EUR_GBP: 11,999
      AUD_USD: 11,999
      USD_JPY: 11,999

   💾 Saved: /content/ml_engine/market_data/MULTI_PAIR_H1.csv

✅ Multi-pair data ready: 59,995 samples


In [65]:
# ============================================================
# 4.5 Data Preview and Validation (Multi-Pair Aware)
# ============================================================
import pandas as pd
import os

# Check if DATA_PATH was set by previous cells
if 'DATA_PATH' not in dir() or DATA_PATH is None:
    # Try to find existing data file
    import glob
    csv_files = glob.glob("/content/ml_engine/market_data/*.csv")
    if csv_files:
        # Prefer multi-pair if available
        multi_path = "/content/ml_engine/market_data/MULTI_PAIR_H1.csv"
        if multi_path in csv_files:
            DATA_PATH = multi_path
        else:
            DATA_PATH = csv_files[0]
        print(f"📁 Using existing data: {DATA_PATH}")
    else:
        print("❌ No data file found. Run cells 4.1-4.4 first to fetch data,")
        print("   or upload CSV files to /content/ml_engine/market_data/")
        DATA_PATH = None

if DATA_PATH:
    # Verify file exists before reading
    if not os.path.exists(DATA_PATH):
        print(f"❌ ERROR: File not found: {DATA_PATH}")
        print("   Please run cells 4.1-4.4 to fetch market data first,")
        print("   or verify the file path is correct.")
        raise FileNotFoundError(f"Data file not found: {DATA_PATH}")
    
    try:
        df = pd.read_csv(DATA_PATH)
        CANDLES = len(df)
        
        print("=" * 70)
        print("📊 DATA PREVIEW")
        print("=" * 70)
        print(f"Shape: {df.shape}")
        print(f"\nColumns: {list(df.columns)}")
        
        # Check if multi-pair data
        if 'pair' in df.columns:
            print("\n🔀 MULTI-PAIR DATASET DETECTED!")
            print(f"\n   Pairs included:")
            for pair in df['pair'].unique():
                pair_count = len(df[df['pair'] == pair])
                print(f"      • {pair}: {pair_count:,} candles")
        
        print(f"\nFirst 5 rows:")
        display(df.head())
        print(f"\nLast 5 rows:")
        display(df.tail())
        print(f"\nStatistics:")
        display(df.describe())
        
        # Check for NaN values
        nan_counts = df.isna().sum()
        if nan_counts.any():
            print(f"\n⚠️ NaN values detected:")
            print(nan_counts[nan_counts > 0])
        else:
            print(f"\n✅ No NaN values in data")
        
        # A100 data sufficiency check
        if 'optimal' in dir():
            sufficiency = CANDLES / optimal['recommended_candles'] * 100
            print(f"\n📊 A100 Data Sufficiency: {sufficiency:.0f}%")
            if sufficiency < 50:
                print(f"   ⚠️ Consider fetching more data for optimal A100 training")
            elif sufficiency >= 100:
                print(f"   ✅ Sufficient data for A100 powerhouse training!")
    
    except Exception as e:
        print(f"❌ ERROR reading file: {e}")
        print(f"   File path: {DATA_PATH}")
        print("   Please run cells 4.1-4.4 to fetch market data.")
        raise
else:
    print("⚠️ DATA_PATH not set. Please run cells 4.1-4.4 to fetch market data first.")


📊 DATA PREVIEW
Shape: (59995, 15)

Columns: ['time', 'open', 'high', 'low', 'close', 'volume', 'instrument', 'pair', 'pair_id', 'norm_open', 'norm_high', 'norm_low', 'norm_close', 'returns', 'log_returns']

🔀 MULTI-PAIR DATASET DETECTED!

   Pairs included:
      • USD_JPY: 11,999 candles
      • EUR_JPY: 11,999 candles
      • EUR_GBP: 11,999 candles
      • GBP_JPY: 11,999 candles
      • AUD_USD: 11,999 candles

First 5 rows:


,time,open,high,low,close,volume,instrument,pair,pair_id,norm_open,norm_high,norm_low,norm_close,returns,log_returns
0,2024-02-02 00:00:00+00:00,146.46400,146.49600,146.27600,146.3900,6107,USD_JPY,USD_JPY,4,1.000505,1.000724,0.999221,1.0,-0.000505,-0.000505
1,2024-02-02 00:00:00+00:00,159.26900,159.29800,159.13800,159.1660,5478,EUR_JPY,EUR_JPY,1,1.000647,1.000829,0.999824,1.0,-0.000640,-0.000641
2,2024-02-02 00:00:00+00:00,0.85312,0.85336,0.85294,0.8531,958,EUR_GBP,EUR_GBP,2,1.000023,1.000305,0.999812,1.0,-0.000070,-0.000070
3,2024-02-02 00:00:00+00:00,186.68100,186.72600,186.51500,186.5770,5742,GBP_JPY,GBP_JPY,0,1.000557,1.000799,0.999668,1.0,-0.000552,-0.000552
4,2024-02-02 00:00:00+00:00,0.65744,0.65872,0.65744,0.6579,1447,AUD_USD,AUD_USD,3,0.999301,1.001246,0.999301,1.0,0.000700,0.000699



Last 5 rows:


,time,open,high,low,close,volume,instrument,pair,pair_id,norm_open,norm_high,norm_low,norm_close,returns,log_returns
59990,2026-01-08 00:00:00+00:00,0.67220,0.67236,0.67167,0.67190,863,AUD_USD,AUD_USD,3,1.000446,1.000685,0.999658,1.0,-0.000446,-0.000446
59991,2026-01-08 00:00:00+00:00,182.98800,183.11400,182.96200,183.08900,4785,EUR_JPY,EUR_JPY,1,0.999448,1.000137,0.999306,1.0,0.000552,0.000552
59992,2026-01-08 00:00:00+00:00,210.89900,211.03000,210.86800,211.00900,6317,GBP_JPY,GBP_JPY,0,0.999479,1.000100,0.999332,1.0,0.000536,0.000536
59993,2026-01-08 00:00:00+00:00,0.86761,0.86774,0.86753,0.86765,496,EUR_GBP,EUR_GBP,2,0.999954,1.000104,0.999862,1.0,0.000058,0.000058
59994,2026-01-08 00:00:00+00:00,156.70000,156.80200,156.67600,156.79000,5435,USD_JPY,USD_JPY,4,0.999426,1.000077,0.999273,1.0,0.000574,0.000574



Statistics:


,open,high,low,close,volume,pair_id,norm_open,norm_high,norm_low,norm_close,returns,log_returns
count,59995.000000,59995.000000,59995.000000,59995.000000,59995.000000,59995.000000,59995.000000,59995.000000,59995.000000,59995.0,59995.000000,59995.000000
mean,103.026398,103.106102,102.941945,103.027986,7184.812984,2.000000,0.999990,1.000699,0.999255,1.0,0.000007,0.000006
std,84.875761,84.940554,84.807086,84.877136,6559.341549,1.414225,0.001099,0.000839,0.000820,0.0,0.001110,0.001111
min,0.594940,0.595620,0.591400,0.594980,8.000000,0.000000,0.981543,1.000000,0.980875,1.0,-0.022140,-0.022389
25%,0.840030,0.840375,0.839660,0.840020,2331.000000,1.000000,0.999526,1.000207,0.999033,1.0,-0.000429,-0.000430
50%,150.448000,150.540000,150.348000,150.455000,5265.000000,2.000000,0.999976,1.000456,0.999485,1.0,0.000016,0.000016
75%,171.954000,172.051500,171.830500,171.958500,10280.500000,3.000000,1.000425,1.000902,0.999759,1.0,0.000470,0.000470
max,212.008000,212.159000,211.936000,212.008000,91059.000000,4.000000,1.022617,1.023105,1.000000,1.0,0.018889,0.018712



✅ No NaN values in data


## 5️⃣ Training Configuration (CUDA-Optimized)

In [83]:
# ============================================================
# 5.1 Training hyperparameters - A100 POWERHOUSE CONFIG
# ============================================================
# Based on NVIDIA A100 Tensor Core optimization research:
# - A100 uses 64-element alignment (not 8!) for maximum efficiency
# - Larger models train efficiently on 80GB VRAM
# - Bigger batch sizes with gradient accumulation for stability

TRAINING_CONFIG = {
    # === MODEL ARCHITECTURE - SCALED FOR A100 ===
    # All dimensions are multiples of 64 for A100 Tensor Core optimization
    "model_type": "ensemble",
    
    # Transformer - 4x LARGER than Mac M1
    "transformer_d_model": 128,      # 32→128 (4x) - more representational capacity
    "transformer_num_heads": 8,      # 4→8 (2x) - more attention patterns
    "transformer_num_layers": 4,     # 2→4 (2x) - deeper network
    "transformer_dff": 512,          # 64→512 (8x) - wider feedforward
    "transformer_dropout": 0.15,     # Slightly less dropout for larger model
    
    # === A100-OPTIMIZED TRAINING ===
    "epochs": 200,
    "batch_size": 256,               # 64→256 (4x) - A100 handles this easily
    "learning_rate": 0.0003,         # Keep proven LR (sqrt scaling not needed with warmup)
    "patience": 25,                  # More patience for larger model
    "seq_len": 128,                  # 64→128 (2x) - more temporal context
    
    # === DIRECTION PREDICTION - PROVEN SETTINGS ===
    "direction_threshold": 0.0015,   # 0.15% - filters noise (KEEP FROM MAC)
    "direction_lookahead": 24,       # 24 bars = 1 day (KEEP FROM MAC)
    
    # === A100 TENSOR CORE ACCELERATION ===
    "use_tf32": True,                # TF32 for float32 ops (1.5x speedup)
    "jit_compile": True,             # XLA compilation (graph optimization)
    "steps_per_execution": 32,       # Reduce Python overhead
    
    # === LEARNING RATE SCHEDULE ===
    "warmup_epochs": 5,              # Warm up LR for stability with large batch
    "use_cosine_decay": True,        # Cosine annealing
    
    # === CONTINUAL LEARNING ===
    "use_ema": True,
    "ema_decay": 0.999,
    "use_ewc": True,
    "ewc_lambda": 1000.0,
    "use_replay_buffer": True,
    "replay_buffer_ratio": 0.10,
    
    # === WALK-FORWARD VALIDATION ===
    "cv_folds": 3,
    "min_train_samples": 4000,
    "test_period": 1000,
    "gap": 24,
    
    # === OVERFITTING PREVENTION ===
    "enable_swa": True,
    "enable_cosine_restarts": True,
    "overfit_threshold": 0.08,
    "critical_threshold": 0.15,
    "max_acceptable_gap": 0.12,
}

print("=" * 70)
print("🚀 A100 POWERHOUSE CONFIG")
print("=" * 70)
print("\n📊 Model Scaling (vs Mac M1):")
print("   ┌──────────────────┬─────────┬─────────┬────────┐")
print("   │ Parameter        │ Mac M1  │ A100    │ Scale  │")
print("   ├──────────────────┼─────────┼─────────┼────────┤")
print("   │ d_model          │ 32      │ 128     │ 4x     │")
print("   │ num_heads        │ 4       │ 8       │ 2x     │")
print("   │ num_layers       │ 2       │ 4       │ 2x     │")
print("   │ dff              │ 64      │ 512     │ 8x     │")
print("   │ seq_len          │ 64      │ 128     │ 2x     │")
print("   │ batch_size       │ 64      │ 256     │ 4x     │")
print("   └──────────────────┴─────────┴─────────┴────────┘")
print(f"\n🎯 Proven Settings (kept from Mac):")
print(f"   • lookahead:  {TRAINING_CONFIG['direction_lookahead']} bars (1-day prediction)")
print(f"   • threshold:  {TRAINING_CONFIG['direction_threshold']*100:.2f}% (noise filter)")
print(f"   • LR:         {TRAINING_CONFIG['learning_rate']} (with warmup)")
print("\n⚡ A100 Optimizations:")
print("   • All dims multiples of 64 (Tensor Core aligned)")
print("   • TF32 enabled for matrix ops")
print("   • XLA compilation for fused kernels")
print("   • XGBoost GPU acceleration (tree_method='gpu_hist')")
print("   • 500 XGBoost trees (vs 100-200 CPU)")
print("\n✅ Expected: ~60-65% balanced accuracy (larger model = better patterns)")

🚀 A100 POWERHOUSE CONFIG

📊 Model Scaling (vs Mac M1):
   ┌──────────────────┬─────────┬─────────┬────────┐
   │ Parameter        │ Mac M1  │ A100    │ Scale  │
   ├──────────────────┼─────────┼─────────┼────────┤
   │ d_model          │ 32      │ 128     │ 4x     │
   │ num_heads        │ 4       │ 8       │ 2x     │
   │ num_layers       │ 2       │ 4       │ 2x     │
   │ dff              │ 64      │ 512     │ 8x     │
   │ seq_len          │ 64      │ 128     │ 2x     │
   │ batch_size       │ 64      │ 256     │ 4x     │
   └──────────────────┴─────────┴─────────┴────────┘

🎯 Proven Settings (kept from Mac):
   • lookahead:  24 bars (1-day prediction)
   • threshold:  0.15% (noise filter)
   • LR:         0.0003 (with warmup)

⚡ A100 Optimizations:
   • All dims multiples of 64 (Tensor Core aligned)
   • TF32 enabled for matrix ops
   • XLA compilation for fused kernels
   • XGBoost GPU acceleration (tree_method='gpu_hist')
   • 500 XGBoost trees (vs 100-200 CPU)

✅ Expected: ~60-

## 6️⃣ Run Training

⚠️ **IMPORTANT**: Before running this section, ensure you have completed:
- ✅ Section 1: Environment Setup & GPU Verification
- ✅ Section 2: Clone Repository & Setup
- ✅ Section 3: OANDA API Configuration (optional for live data)
- ✅ Section 4: Fetch Market Data (creates DATA_PATH)
- ✅ Section 5: Training Configuration

If you skip Section 4, you'll see an "Unable to read file" error.


In [84]:
# ============================================================
# 6.0 Pre-Training Validation
# ============================================================
# Verify all prerequisites are met before training starts

import os
from rich.console import Console
from rich.panel import Panel

console = Console()

console.print(Panel("[bold cyan]Pre-Training Checklist[/bold cyan]"))

errors = []
warnings = []

# Check 1: DATA_PATH is set
if 'DATA_PATH' not in dir() or DATA_PATH is None:
    errors.append("❌ DATA_PATH not set - Run cells 4.1-4.4 to fetch market data")
else:
    console.print(f"  ✅ DATA_PATH set: {DATA_PATH}")
    
    # Check 2: File exists
    if not os.path.exists(DATA_PATH):
        errors.append(f"❌ Data file not found: {DATA_PATH}")
    else:
        file_size = os.path.getsize(DATA_PATH) / (1024 * 1024)  # MB
        console.print(f"  ✅ Data file exists ({file_size:.1f} MB)")

# Check 3: TRAINING_CONFIG is defined
if 'TRAINING_CONFIG' not in dir():
    errors.append("❌ TRAINING_CONFIG not defined - Run cell 5.1 (Training Configuration)")
else:
    console.print(f"  ✅ TRAINING_CONFIG defined")

# Check 4: GPU available
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    warnings.append("⚠️ No GPU detected - Training will be slow on CPU")
else:
    console.print(f"  ✅ GPU available: {len(gpus)} device(s)")

# Check 5: Custom losses registered
if 'CUSTOM_OBJECTS' not in dir():
    warnings.append("⚠️ CUSTOM_OBJECTS not defined - Run cell 1.4 (Register Custom Losses)")
else:
    console.print(f"  ✅ Custom losses registered")

# Display results
if errors:
    console.print("\n[red bold]ERRORS - Cannot proceed with training:[/red bold]")
    for error in errors:
        console.print(f"  {error}")
    console.print("\n[yellow]Please fix the errors above and run this cell again.[/yellow]")
    raise RuntimeError("Pre-training validation failed. See errors above.")

if warnings:
    console.print("\n[yellow]WARNINGS:[/yellow]")
    for warning in warnings:
        console.print(f"  {warning}")

if not errors:
    console.print("\n[green bold]✅ All checks passed! Ready to train.[/green bold]")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Pre-Training Checklist                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ DATA_PATH set: /content/ml_engine/market_data/MULTI_PAIR_H1.csv

✅ Data file exists (10.6 MB)

✅ TRAINING_CONFIG defined

✅ GPU available: 1 device(s)

✅ Custom losses registered

✅ All checks passed! Ready to train.

In [85]:
# ============================================================
# 6.1 Import training modules
# ============================================================
import os
import sys
import logging

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Add repo to path
sys.path.insert(0, '/content/ml_engine')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("📦 Importing training modules...")

import numpy as np
import pandas as pd
import tensorflow as tf

# Import trainers (correct class names!)
from modular_trainers import (
    TrainerConfig,
    TransformerDirectionTrainer,
    XGBoostTrainer,          # NOT XGBoostMomentumTrainer
    RandomForestTrainer,     # NOT RandomForestRiskTrainer
    RidgeTrainer,            # NOT RidgeConfidenceTrainer
    OverfitPreventionCallback,
)

# Import data loaders (correct function names!)
from modular_data_loaders import (
    compute_normalized_features,
    load_direction_data,      # NOT prepare_direction_data
    load_xgboost_data,        # NOT prepare_momentum_data
    load_rf_data,             # NOT prepare_risk_data
    load_ridge_data,          # NOT prepare_confidence_data
)

# Import feature engineering
from feature_engineering import FeatureEngineering

print("✅ All modules imported successfully!")
print(f"   TensorFlow: {tf.__version__}")
print(f"   GPU devices: {tf.config.list_physical_devices('GPU')}")

📦 Importing training modules...
✅ All modules imported successfully!
   TensorFlow: 2.19.0
   GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [86]:
# ============================================================
# 6.2 Prepare training data
# ============================================================
from rich.console import Console
from rich.panel import Panel
import os

console = Console()

console.print(Panel("[bold blue]Step 1: Data Preparation[/bold blue]"))

# Verify DATA_PATH is set and file exists
if 'DATA_PATH' not in dir() or DATA_PATH is None:
    console.print("[red]❌ ERROR: DATA_PATH not set![/red]")
    console.print("[yellow]Please run cells 4.1-4.4 to fetch market data first.[/yellow]")
    raise ValueError("DATA_PATH not set. Run data fetching cells first.")

if not os.path.exists(DATA_PATH):
    console.print(f"[red]❌ ERROR: File not found: {DATA_PATH}[/red]")
    console.print("[yellow]Please run cells 4.1-4.4 to fetch market data.[/yellow]")
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

# Load data
console.print(f"  📂 Loading data from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)

if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

# Rename columns to lowercase
df.columns = [c.lower() for c in df.columns]

console.print(f"  📊 Loaded {len(df)} candles")
console.print(f"  📅 Date range: {df.index.min()} to {df.index.max()}")

# Compute normalized features
console.print("  🔧 Computing normalized features...")
df = compute_normalized_features(df)

# Add technical indicators
fe = FeatureEngineering()
df = fe.add_technical_indicators(df)

# Fill NaN values
df = df.ffill().bfill()

# Drop remaining NaN rows
df = df.dropna()

console.print(f"  ✅ Features computed: {len(df.columns)} columns")
console.print(f"  ✅ Clean rows: {len(df)}")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 1: Data Preparation                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📂 Loading data from: /content/ml_engine/market_data/MULTI_PAIR_H1.csv

📊 Loaded 59995 candles

📅 Date range: 2024-02-02 00:00:00+00:00 to 2026-01-08 00:00:00+00:00

🔧 Computing normalized features...

✅ Features computed: 126 columns

✅ Clean rows: 59995

In [87]:
# ============================================================
# 6.3 Prepare model-specific datasets
# ============================================================
console.print(Panel("[bold blue]Step 2: Prepare Model-Specific Data[/bold blue]"))

SEQ_LEN = TRAINING_CONFIG["seq_len"]           # 128 (2x Mac for more context)
LOOKAHEAD = TRAINING_CONFIG["direction_lookahead"]  # 24 bars (1 day) - proven setting
THRESHOLD = TRAINING_CONFIG["direction_threshold"]   # 0.15% - filters noise

# Direction data (Transformer) - uses load_direction_data
console.print("  📊 Preparing Direction data (Transformer)...")
console.print(f"     seq_len={SEQ_LEN} (A100: 2x temporal context)")
console.print(f"     lookahead={LOOKAHEAD} bars, threshold={THRESHOLD*100:.2f}%")

direction_data = load_direction_data(
    df, 
    split=(0.8, 0.1, 0.1), 
    lookahead=LOOKAHEAD,     # 24 bars = 1 day prediction
    threshold=THRESHOLD       # 0.15% filters noise
)
console.print(f"     X_train: {direction_data['X_train'].shape}")
console.print(f"     y_train: {direction_data['y_train'].shape}")

# Check class distribution
y_train_direction = direction_data['y_train']
n_up = (y_train_direction == 1).sum()
n_down = (y_train_direction == 0).sum()
n_unclear = ((y_train_direction != 0) & (y_train_direction != 1)).sum()
total = len(y_train_direction)

console.print(f"     📊 Class distribution: UP={n_up} ({100*n_up/total:.1f}%), DOWN={n_down} ({100*n_down/total:.1f}%)")
if n_unclear > 0:
    console.print(f"     ⚠️ Unclear samples filtered: {n_unclear} ({100*n_unclear/total:.1f}%)")

# Check imbalance ratio
if n_up > 0 and n_down > 0:
    imbalance = max(n_up, n_down) / min(n_up, n_down)
    if imbalance > 2.0:
        console.print(f"     ⚠️ High imbalance: {imbalance:.2f}x - class weights will be applied")
    else:
        console.print(f"     ✅ Balanced classes (imbalance: {imbalance:.2f}x)")

# Momentum data (XGBoost) - uses load_xgboost_data
console.print("  📊 Preparing Momentum data (XGBoost)...")
momentum_data = load_xgboost_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {momentum_data['X_train'].shape}")

# Risk data (Random Forest) - uses load_rf_data
console.print("  📊 Preparing Risk data (Random Forest)...")
risk_data = load_rf_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {risk_data['X_train'].shape}")

# Confidence data (Ridge) - uses load_ridge_data
console.print("  📊 Preparing Confidence data (Ridge)...")
confidence_data = load_ridge_data(df, split=(0.8, 0.1, 0.1))
console.print(f"     X_train: {confidence_data['X_train'].shape}")

console.print("\n✅ All datasets prepared (A100 Powerhouse config)!")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 2: Prepare Model-Specific Data                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Preparing Direction data (Transformer)...

seq_len=128 (A100: 2x temporal context)

lookahead=24 bars, threshold=0.15%

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


...

📊 Preparing Risk data (Random Forest)...

X_train: (47980, 18)

📊 Preparing Confidence data (Ridge)...

X_train: (47988, 14)

✅ All datasets prepared (A100 Powerhouse config)!

In [88]:
# ============================================================
# 6.3.1 FIX: Scale Direction Data & Remove Constant Features
# ============================================================
# The data loader doesn't scale features - fix it here before training

from sklearn.preprocessing import RobustScaler

console.print(Panel("[bold yellow]🔧 FIXING DIRECTION DATA SCALING[/bold yellow]"))

# Check current state
x_max_before = np.max(np.abs(direction_data['X_train']))
console.print(f"  Before: max={x_max_before:.2e}")

# Step 1: Remove constant features
feature_stds = np.std(direction_data['X_train'], axis=0)
valid_mask = feature_stds > 1e-6
n_constant = np.sum(~valid_mask)

if n_constant > 0:
    console.print(f"  Removing {n_constant} constant features...")
    direction_data['X_train'] = direction_data['X_train'][:, valid_mask]
    direction_data['X_val'] = direction_data['X_val'][:, valid_mask]
    direction_data['X_test'] = direction_data['X_test'][:, valid_mask]
    # Update feature names if present
    if 'feature_names' in direction_data and direction_data['feature_names']:
        direction_data['feature_names'] = [f for f, v in zip(direction_data['feature_names'], valid_mask) if v]

# Step 2: Scale features using RobustScaler (fit on train only)
scaler = RobustScaler()
direction_data['X_train'] = scaler.fit_transform(direction_data['X_train']).astype(np.float32)
direction_data['X_val'] = scaler.transform(direction_data['X_val']).astype(np.float32)
direction_data['X_test'] = scaler.transform(direction_data['X_test']).astype(np.float32)

# Step 3: Clip extreme values
clip_value = 10.0
direction_data['X_train'] = np.clip(direction_data['X_train'], -clip_value, clip_value)
direction_data['X_val'] = np.clip(direction_data['X_val'], -clip_value, clip_value)
direction_data['X_test'] = np.clip(direction_data['X_test'], -clip_value, clip_value)

# Verify fix
x_max_after = np.max(np.abs(direction_data['X_train']))
x_mean_after = np.mean(np.abs(direction_data['X_train']))

console.print(f"  After:  max={x_max_after:.2f}, mean_abs={x_mean_after:.4f}")
console.print(f"  Shape:  {direction_data['X_train'].shape}")
console.print(f"[green]  ✅ Direction data fixed![/green]")

# Save scaler for inference
direction_data['scaler'] = scaler

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🔧 FIXING DIRECTION DATA SCALING                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Before: max=1.00e+01

After:  max=10.00, mean_abs=0.7099

Shape:  (47976, 58)

  ✅ Direction data fixed!

In [89]:
# ============================================================
# 6.4 Train Transformer (Direction Predictor) - ANTI-COLLAPSE VERSION
# ============================================================
# Uses Focal Loss + Class Weights + Anti-Collapse Regularization
# to prevent model from predicting all one class

console.print(Panel("[bold green]Step 3/6: Training Transformer (Direction) - ANTI-COLLAPSE[/bold green]"))

import tensorflow as tf
from tensorflow import keras

# === FOCAL LOSS: Focuses on hard examples, reduces easy example weight ===
def focal_loss(gamma=2.0, alpha=0.5):
    """
    Focal Loss: FL(p_t) = -alpha * (1-p_t)^gamma * log(p_t)
    - gamma=2.0: Reduces weight of easy examples (high confidence)
    - alpha=0.5: Balance between classes (will be overridden by class_weight)
    """
    def focal_loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        # Binary focal loss
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        ce = -tf.math.log(pt)
        focal_weight = tf.pow(1 - pt, gamma)
        
        return tf.reduce_mean(alpha * focal_weight * ce)
    return focal_loss_fn

# === ANTI-COLLAPSE LOSS: Penalizes predictions that are too one-sided ===
def anti_collapse_loss(base_loss_fn, collapse_penalty=0.1):
    """
    Adds a penalty if mean prediction is too far from 0.5 (balanced)
    This prevents the model from predicting all UP or all DOWN
    """
    def combined_loss(y_true, y_pred):
        # Base loss
        loss = base_loss_fn(y_true, y_pred)
        
        # Collapse penalty: penalize if mean prediction is far from 0.5
        mean_pred = tf.reduce_mean(y_pred)
        collapse_term = collapse_penalty * tf.square(mean_pred - 0.5)
        
        return loss + collapse_term
    return combined_loss

# === BUILD MODEL WITH ANTI-COLLAPSE ARCHITECTURE ===
def build_anti_collapse_transformer(n_features, config):
    """
    Transformer with:
    - Output bias initialized to log(down/up) for balanced starting point
    - Dropout before output to prevent overconfidence
    """
    inputs = keras.Input(shape=(n_features,))
    
    # Input projection
    x = keras.layers.Dense(config.transformer_d_model, activation='relu')(inputs)
    x = keras.layers.Dropout(config.transformer_dropout)(x)
    x = keras.layers.LayerNormalization()(x)
    
    # Transformer-style blocks (simplified for 2D input)
    for _ in range(config.transformer_num_layers):
        # Self-attention approximation via dense layers
        attn = keras.layers.Dense(config.transformer_d_model, activation='relu')(x)
        attn = keras.layers.Dropout(config.transformer_dropout)(attn)
        x = keras.layers.Add()([x, attn])
        x = keras.layers.LayerNormalization()(x)
        
        # FFN
        ffn = keras.layers.Dense(config.transformer_dff, activation='relu')(x)
        ffn = keras.layers.Dropout(config.transformer_dropout)(ffn)
        ffn = keras.layers.Dense(config.transformer_d_model)(ffn)
        x = keras.layers.Add()([x, ffn])
        x = keras.layers.LayerNormalization()(x)
    
    # Output with balanced bias initialization
    x = keras.layers.Dropout(0.3)(x)  # Extra dropout before output
    
    # Initialize output bias to 0 (balanced 50/50 starting point)
    outputs = keras.layers.Dense(
        1, 
        activation='sigmoid',
        bias_initializer=keras.initializers.Constant(0.0)  # Start at 50/50
    )(x)
    
    return keras.Model(inputs, outputs)

# === PREPARE DATA ===
X_train = direction_data['X_train']
y_train = direction_data['y_train']
X_val = direction_data['X_val']
y_val = direction_data['y_val']

# Filter out unclear samples (y=0.5)
train_mask = (y_train == 0) | (y_train == 1)
val_mask = (y_val == 0) | (y_val == 1)

X_train_f = X_train[train_mask]
y_train_f = y_train[train_mask]
X_val_f = X_val[val_mask]
y_val_f = y_val[val_mask]

console.print(f"  📊 Training data: {len(X_train_f):,} samples")
console.print(f"  📊 Validation data: {len(X_val_f):,} samples")

# === COMPUTE CLASS WEIGHTS ===
n_up = (y_train_f == 1).sum()
n_down = (y_train_f == 0).sum()
total = n_up + n_down

# Inverse frequency weighting
weight_up = total / (2 * n_up) if n_up > 0 else 1.0
weight_down = total / (2 * n_down) if n_down > 0 else 1.0

class_weight = {0: weight_down, 1: weight_up}
console.print(f"  ⚖️ Class weights: DOWN={weight_down:.3f}, UP={weight_up:.3f}")

# === CREATE CONFIG ===
from dataclasses import dataclass

@dataclass
class SimpleConfig:
    transformer_d_model: int = TRAINING_CONFIG["transformer_d_model"]
    transformer_num_heads: int = TRAINING_CONFIG["transformer_num_heads"]
    transformer_num_layers: int = TRAINING_CONFIG["transformer_num_layers"]
    transformer_dff: int = TRAINING_CONFIG["transformer_dff"]
    transformer_dropout: float = 0.2  # Increased dropout to prevent collapse

simple_config = SimpleConfig()

# === BUILD MODEL ===
n_features = X_train_f.shape[1]
model = build_anti_collapse_transformer(n_features, simple_config)

console.print(f"  🏗️ Model built: {model.count_params():,} parameters")

# === COMPILE WITH ANTI-COLLAPSE LOSS ===
base_loss = focal_loss(gamma=2.0, alpha=0.5)
combined_loss = anti_collapse_loss(base_loss, collapse_penalty=0.2)

# Use lower LR to prevent jumping to degenerate solutions
optimizer = keras.optimizers.Adam(learning_rate=0.0001)  # Reduced from 0.0003

model.compile(
    optimizer=optimizer,
    loss=combined_loss,
    metrics=['accuracy']
)

# === CALLBACKS ===
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=8,
        mode='max',
        min_lr=1e-6,
        verbose=1
    ),
]

# === CUSTOM TRAINING CALLBACK TO MONITOR COLLAPSE ===
class CollapseMonitor(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(X_val_f[:1000], verbose=0)
        up_pct = (preds > 0.5).mean() * 100
        down_pct = 100 - up_pct
        
        if epoch % 5 == 0:
            console.print(f"  Epoch {epoch+1:3d} | val_acc={logs.get('val_accuracy', 0):.1%} | "
                         f"preds: {up_pct:.0f}% UP, {down_pct:.0f}% DOWN")
        
        # Warn on collapse
        if up_pct > 90 or down_pct > 90:
            console.print(f"[red]  ⚠️ Epoch {epoch+1}: COLLAPSE WARNING - {up_pct:.0f}% UP[/red]")

callbacks.append(CollapseMonitor())

# === TRAIN ===
console.print(f"\n🚀 Training with ANTI-COLLAPSE settings:")
console.print(f"   • Focal Loss (gamma=2.0) - focuses on hard examples")
console.print(f"   • Anti-collapse penalty - prevents one-sided predictions")
console.print(f"   • Class weights - balances UP/DOWN")
console.print(f"   • Lower LR (0.0001) - prevents degenerate solutions")
console.print(f"   • Higher dropout (0.2) - prevents overconfidence\n")

history = model.fit(
    X_train_f, y_train_f,
    validation_data=(X_val_f, y_val_f),
    epochs=100,  # Reduced epochs since we're using early stopping
    batch_size=TRAINING_CONFIG["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=0  # CollapseMonitor handles output
)

# === EVALUATE ===
y_pred = model.predict(X_val_f, verbose=0)
y_pred_classes = (y_pred > 0.5).astype(int).flatten()

# Calculate balanced accuracy
from sklearn.metrics import balanced_accuracy_score, accuracy_score

val_accuracy = accuracy_score(y_val_f, y_pred_classes)
val_balanced = balanced_accuracy_score(y_val_f, y_pred_classes)

# Per-class accuracy
up_mask = y_val_f == 1
down_mask = y_val_f == 0
up_acc = (y_pred_classes[up_mask] == 1).mean() if up_mask.sum() > 0 else 0
down_acc = (y_pred_classes[down_mask] == 0).mean() if down_mask.sum() > 0 else 0

# Prediction distribution
up_pct = (y_pred > 0.5).mean() * 100
down_pct = 100 - up_pct

console.print(f"\n✅ Training Complete!")
console.print(f"   Val Accuracy:     {val_accuracy:.1%}")
console.print(f"   Balanced Acc:     {val_balanced:.1%}")
console.print(f"   ↳ UP accuracy:    {up_acc:.1%}")
console.print(f"   ↳ DOWN accuracy:  {down_acc:.1%}")
console.print(f"   Prediction dist:  {up_pct:.0f}% UP, {down_pct:.0f}% DOWN")

# Store results for later cells
transformer_result = {
    'val_accuracy': val_accuracy,
    'val_balanced_accuracy': val_balanced,
    'val_up_accuracy': up_acc,
    'val_down_accuracy': down_acc,
}

# Store model for saving
transformer_trainer = type('obj', (object,), {
    'model': model,
    'save': lambda self, path, **kwargs: model.save(path),
    'is_trained': True
})()

if val_balanced > 0.52:
    console.print(f"[green]   🎉 Model is learning! Balanced acc > 52%[/green]")
elif val_balanced > 0.48:
    console.print(f"[yellow]   ⚠️ Marginal improvement over random[/yellow]")
else:
    console.print(f"[red]   ❌ Model may still be collapsed or random[/red]")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 3/6: Training Transformer (Direction) - ANTI-COLLAPSE                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

📊 Training data: 42,299 samples

📊 Validation data: 5,075 samples

⚖️ Class weights: DOWN=1.006, UP=0.994

🏗️ Model built: 602,881 parameters

...

Balanced Acc:     78.2%

↳ UP accuracy:    69.8%

↳ DOWN accuracy:  86.6%

Prediction dist:  42% UP, 58% DOWN

   🎉 Model is learning! Balanced acc > 52%

In [90]:
# ============================================================
# 6.4.1 ⚠️ VERIFY RESULTS ARE REAL (Run AFTER training!)
# ============================================================
# This cell verifies the trained model isn't benefiting from data leakage
# Run this AFTER training completes to validate results

from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix
import numpy as np

console.print(Panel("[bold red]⚠️ RESULTS VERIFICATION[/bold red]"))

# === TEST 1: Check trained model's actual predictions ===
console.print("\n[bold]Test 1: Verify Trained Model Predictions[/bold]")

y_pred_proba = model.predict(X_val_f, verbose=0)
y_pred_class = (y_pred_proba > 0.5).astype(int).flatten()

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_val_f, y_pred_class).ravel()
console.print(f"  Confusion Matrix:")
console.print(f"    True Negatives (correct DOWN):  {tn}")
console.print(f"    False Positives (wrong UP):     {fp}")
console.print(f"    False Negatives (wrong DOWN):   {fn}")
console.print(f"    True Positives (correct UP):    {tp}")

actual_accuracy = accuracy_score(y_val_f, y_pred_class)
actual_balanced = balanced_accuracy_score(y_val_f, y_pred_class)

console.print(f"\n  Actual Accuracy: {actual_accuracy:.1%}")
console.print(f"  Actual Balanced: {actual_balanced:.1%}")

# === TEST 2: Shuffled Label Test (detects feature leakage) ===
console.print("\n[bold]Test 2: Shuffled Labels Test (Leakage Detection)[/bold]")
console.print("  Training NEW model on randomly shuffled labels...")
console.print("  If it learns anything, features contain future info!")

# Create fresh model
from tensorflow import keras
model_shuffle_test = build_anti_collapse_transformer(n_features, simple_config)
model_shuffle_test.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Shuffle training labels
y_train_shuffled = y_train_f.copy()
np.random.seed(42)  # Reproducible
np.random.shuffle(y_train_shuffled)

# Quick train
model_shuffle_test.fit(
    X_train_f, y_train_shuffled,
    epochs=15,
    batch_size=256,
    verbose=0,
    validation_data=(X_val_f, y_val_f)
)

# Evaluate on REAL (unshuffled) validation labels
shuffle_preds = model_shuffle_test.predict(X_val_f, verbose=0)
shuffle_classes = (shuffle_preds > 0.5).astype(int).flatten()
shuffle_acc = accuracy_score(y_val_f, shuffle_classes)

if shuffle_acc > 0.55:
    console.print(f"[red]  ❌ LEAKAGE DETECTED![/red]")
    console.print(f"[red]     Shuffled-label model accuracy: {shuffle_acc:.1%}[/red]")
    console.print(f"[red]     This should be ~50% if features are clean[/red]")
    console.print(f"[red]     Your 78% accuracy is FAKE - features contain the answer![/red]")
    LEAKAGE_DETECTED = True
else:
    console.print(f"[green]  ✅ No leakage detected[/green]")
    console.print(f"     Shuffled-label accuracy: {shuffle_acc:.1%} (expected ~50%)")
    LEAKAGE_DETECTED = False

# === TEST 3: Forward Walk Test (most rigorous) ===
console.print("\n[bold]Test 3: Forward Walk Prediction Test[/bold]")
console.print("  Testing on truly unseen future data (last 10% of val set)...")

# Use last portion of validation as "future"
future_size = len(X_val_f) // 10
X_future = X_val_f[-future_size:]
y_future = y_val_f[-future_size:]

future_preds = model.predict(X_future, verbose=0)
future_classes = (future_preds > 0.5).astype(int).flatten()
future_acc = accuracy_score(y_future, future_classes)
future_balanced = balanced_accuracy_score(y_future, future_classes)

console.print(f"  Future accuracy: {future_acc:.1%}")
console.print(f"  Future balanced: {future_balanced:.1%}")

# === SUMMARY ===
console.print("\n" + "="*70)
console.print("[bold]VERIFICATION SUMMARY[/bold]")
console.print("="*70)

if LEAKAGE_DETECTED:
    console.print(f"\n[bold red]❌ RESULTS ARE NOT REAL[/bold red]")
    console.print(f"   The high accuracy ({actual_accuracy:.1%}) is due to data leakage.")
    console.print(f"\n   RECOMMENDATIONS:")
    console.print(f"   1. Set MULTI_PAIR_MODE = False in cell 14")
    console.print(f"   2. Re-run data preparation (cells 14-17)")
    console.print(f"   3. Re-train - expect realistic 52-58% accuracy")
elif actual_balanced > 0.65:
    console.print(f"\n[bold yellow]⚠️ SUSPICIOUSLY HIGH ({actual_balanced:.1%})[/bold yellow]")
    console.print(f"   No obvious leakage detected, but >65% is unusual for forex.")
    console.print(f"   Consider retraining on single pair to verify.")
else:
    console.print(f"\n[bold green]✅ RESULTS APPEAR GENUINE[/bold green]")
    console.print(f"   Balanced Accuracy: {actual_balanced:.1%}")
    console.print(f"   This is {'good' if actual_balanced > 0.54 else 'reasonable'} for forex H1 prediction.")

# Save verification results
verification_result = {
    'actual_accuracy': actual_accuracy,
    'actual_balanced': actual_balanced,
    'shuffle_accuracy': shuffle_acc,
    'future_accuracy': future_acc,
    'leakage_detected': LEAKAGE_DETECTED,
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}
}
console.print(f"\n📊 Verification saved to: verification_result")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ⚠️ RESULTS VERIFICATION                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Test 1: Verify Trained Model Predictions

Confusion Matrix:

True Negatives (correct DOWN):  2198

False Positives (wrong UP):     340

...

======================================================================

⚠️ SUSPICIOUSLY HIGH (78.2%)

No obvious leakage detected, but >65% is unusual for forex.

Consider retraining on single pair to verify.

📊 Verification saved to: verification_result

In [91]:
# ============================================================
# 6.5 Train XGBoost (Momentum Analyzer) - A100 GPU-ACCELERATED
# ============================================================
console.print(Panel("[bold green]Step 4/6: Training XGBoost (Momentum) - A100 GPU MODE[/bold green]"))

# ===== PRE-TRAINING DATA VALIDATION =====
console.print("  🔍 Validating momentum data...")

# Check data shapes
X_train_xgb = momentum_data['X_train']
y_train_xgb = momentum_data['y_train']
X_val_xgb = momentum_data['X_val']
y_val_xgb = momentum_data['y_val']

console.print(f"     X_train: {X_train_xgb.shape}")
console.print(f"     y_train: {y_train_xgb.shape}")

# Check target distribution (should have 2 columns: momentum_score, acceleration)
if y_train_xgb.ndim == 2 and y_train_xgb.shape[1] == 2:
    momentum_scores = y_train_xgb[:, 0]
    accelerations = y_train_xgb[:, 1]
    
    console.print(f"\n  📊 Target Distribution:")
    console.print(f"     Momentum Score Range: [{momentum_scores.min():.4f}, {momentum_scores.max():.4f}]")
    console.print(f"     Momentum Mean: {momentum_scores.mean():.4f}")
    console.print(f"     Momentum Std: {momentum_scores.std():.4f}")
    
    # Check acceleration balance
    accel_pos = (accelerations == 1).sum()
    accel_neg = (accelerations == 0).sum()
    console.print(f"     Acceleration: {accel_pos} growing ({100*accel_pos/len(accelerations):.1f}%), {accel_neg} shrinking ({100*accel_neg/len(accelerations):.1f}%)")
    
    # Data quality warnings
    if momentum_scores.mean() < 0.1:
        console.print(f"     [yellow]⚠️ Very low momentum scores - market may be range-bound[/yellow]")
    elif momentum_scores.mean() > 0.5:
        console.print(f"     [yellow]⚠️ Very high momentum scores - highly trending market[/yellow]")
    
    imbalance = max(accel_pos, accel_neg) / max(min(accel_pos, accel_neg), 1)
    if imbalance > 2.0:
        console.print(f"     [yellow]⚠️ Acceleration imbalance: {imbalance:.2f}x - XGBoost will auto-balance[/yellow]")
else:
    console.print(f"     [red]❌ ERROR: Expected 2D target with 2 columns, got shape {y_train_xgb.shape}[/red]")
    raise ValueError(f"Invalid XGBoost target shape: {y_train_xgb.shape}")

# ===== A100 GPU-ACCELERATED TRAINING =====
console.print(f"\n  🚀 Training XGBoost with A100 GPU acceleration...")
console.print(f"     • GPU tree_method='gpu_hist' (10-20x faster)")
console.print(f"     • n_estimators=500 (vs 100 CPU default)")
console.print(f"     • max_depth=8 (vs 6 CPU default)")

# A100-optimized config for XGBoost
config = TrainerConfig(
    epochs=TRAINING_CONFIG["epochs"],
    batch_size=TRAINING_CONFIG["batch_size"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    patience=TRAINING_CONFIG["patience"],
    # A100-specific XGBoost settings
    xgb_n_estimators=500,      # 5x more trees (A100 handles this easily)
    xgb_max_depth=8,           # Deeper trees for complex patterns
    xgb_learning_rate=0.05,    # Lower LR with more trees
    use_gpu=True,              # Enable GPU acceleration
)

xgb_trainer = XGBoostTrainer(config)

import time
start_time = time.time()

xgb_result = xgb_trainer.train(
    X_train=X_train_xgb,
    y_train=y_train_xgb,
    X_val=X_val_xgb,
    y_val=y_val_xgb,
    feature_names=momentum_data.get('feature_names'),
    momentum_norm_factor=momentum_data.get('momentum_norm_factor'),
)

training_time = time.time() - start_time

# ===== POST-TRAINING VALIDATION =====
console.print(f"\n  📊 XGBoost Results (A100 GPU):")
console.print(f"     Training Time: {training_time:.1f}s (A100 GPU acceleration)")

momentum_mae = xgb_result.get('momentum_mae', 0)
accel_acc = xgb_result.get('acceleration_accuracy', 0)

console.print(f"     Momentum MAE: {momentum_mae:.4f}")
console.print(f"     Acceleration Accuracy: {accel_acc:.1%}")

# Quality assessment with context-aware thresholds
if momentum_mae < 0.05:
    console.print(f"     [bold green]✅ EXCELLENT momentum prediction (MAE < 0.05)[/bold green]")
elif momentum_mae < 0.10:
    console.print(f"     [green]✅ GOOD momentum prediction (MAE < 0.10)[/green]")
elif momentum_mae < 0.15:
    console.print(f"     [yellow]⚠️ ACCEPTABLE momentum prediction (MAE < 0.15)[/yellow]")
else:
    console.print(f"     [red]❌ POOR momentum prediction (MAE > 0.15) - Consider more data or features[/red]")

if accel_acc > 0.65:
    console.print(f"     [bold green]✅ EXCELLENT acceleration prediction (>{65}%)[/bold green]")
elif accel_acc > 0.55:
    console.print(f"     [green]✅ GOOD acceleration prediction (>{55}%)[/green]")
else:
    console.print(f"     [yellow]⚠️ WEAK acceleration prediction (<{55}%) - May struggle with trend changes[/yellow]")

console.print(f"\n✅ XGBoost training complete (A100-accelerated)!")
console.print(f"   💡 A100 advantage: {500/100}x more trees trained in similar time vs CPU")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Step 4/6: Training XGBoost (Momentum) - A100 GPU MODE                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔍 Validating momentum data...

X_train: (47984, 19)

y_train: (47984, 2)

📊 Target Distribution:

...

🚀 Training XGBoost with A100 GPU acceleration...

• GPU tree_method='gpu_hist' (10-20x faster)

• n_estimators=500 (vs 100 CPU default)

• max_depth=8 (vs 6 CPU default)

TypeError: TrainerConfig.__init__() got an unexpected keyword argument 'use_gpu'

In [ ]:
# ============================================================
# 6.6 Train Random Forest (Risk Assessor) - A100 OPTIMIZED
# ============================================================
console.print(Panel("[bold green]Step 5/6: Training Random Forest (Risk) - A100 SCALING[/bold green]"))

# ===== PRE-TRAINING DATA VALIDATION =====
console.print("  🔍 Validating risk data...")

# Check data shapes
X_train_rf = risk_data['X_train']
y_train_rf = risk_data['y_train']
X_val_rf = risk_data['X_val']
y_val_rf = risk_data['y_val']

console.print(f"     X_train: {X_train_rf.shape}")
console.print(f"     y_train: {y_train_rf.shape}")

# Check target distribution (should have 2 columns: drawdown_pct, streak_prob)
if y_train_rf.ndim == 2 and y_train_rf.shape[1] == 2:
    drawdown_pcts = y_train_rf[:, 0]
    streak_probs = y_train_rf[:, 1]
    
    console.print(f"\n  📊 Target Distribution:")
    console.print(f"     Drawdown % Range: [{drawdown_pcts.min()*100:.3f}%, {drawdown_pcts.max()*100:.3f}%]")
    console.print(f"     Drawdown Mean: {drawdown_pcts.mean()*100:.3f}% ({drawdown_pcts.mean()*10000:.1f} bps)")
    console.print(f"     Drawdown Std: {drawdown_pcts.std()*100:.3f}%")
    
    console.print(f"\n     Streak Probability Range: [{streak_probs.min():.3f}, {streak_probs.max():.3f}]")
    console.print(f"     Streak Mean: {streak_probs.mean():.3f}")
    console.print(f"     Streak Std: {streak_probs.std():.3f}")
    
    # Data quality checks
    if drawdown_pcts.min() < 0:
        console.print(f"     [red]❌ ERROR: Negative drawdown values detected![/red]")
        raise ValueError("Drawdown percentages cannot be negative")
    
    if drawdown_pcts.max() > 0.20:
        console.print(f"     [yellow]⚠️ Very high drawdown values (>{20}%) - extreme volatility market[/yellow]")
    
    if not (0 <= streak_probs.min() and streak_probs.max() <= 1):
        console.print(f"     [red]❌ ERROR: Streak probabilities outside 0-1 range![/red]")
        raise ValueError(f"Streak probs must be in [0,1], got [{streak_probs.min():.3f}, {streak_probs.max():.3f}]")
    
    # Check for constant values (no variation = can't learn)
    if drawdown_pcts.std() < 1e-6:
        console.print(f"     [red]❌ ERROR: Drawdown values are constant - no variation to learn![/red]")
        raise ValueError("Drawdown target has no variation")
    
    if streak_probs.std() < 1e-6:
        console.print(f"     [yellow]⚠️ Streak probabilities are constant - limited learning possible[/yellow]")
    
else:
    console.print(f"     [red]❌ ERROR: Expected 2D target with 2 columns, got shape {y_train_rf.shape}[/red]")
    raise ValueError(f"Invalid Random Forest target shape: {y_train_rf.shape}")

# ===== A100-OPTIMIZED TRAINING =====
console.print(f"\n  🌲 Training Random Forest with A100 scaling...")
console.print(f"     • n_estimators=1000 (vs 200 CPU default)")
console.print(f"     • max_depth=15 (vs 10 CPU default)")
console.print(f"     • Parallel tree building across all CPU cores")
console.print(f"     • A100's 80GB RAM handles massive ensemble easily")

# A100-optimized config for Random Forest
config_rf = TrainerConfig(
    epochs=TRAINING_CONFIG["epochs"],
    batch_size=TRAINING_CONFIG["batch_size"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    patience=TRAINING_CONFIG["patience"],
    # A100-specific RF settings (leverage massive RAM)
    rf_n_estimators=1000,      # 5x more trees for better generalization
    rf_max_depth=15,           # Deeper trees capture complex risk patterns
    rf_min_samples_leaf=5,     # Lower for more granular splits
)

rf_trainer = RandomForestTrainer(config_rf)

import time
start_time = time.time()

rf_result = rf_trainer.train(
    X_train=X_train_rf,
    y_train=y_train_rf,
    X_val=X_val_rf,
    y_val=y_val_rf,
    feature_names=risk_data.get('feature_names'),
)

training_time = time.time() - start_time

# ===== POST-TRAINING VALIDATION =====
console.print(f"\n  📊 Random Forest Results (A100-Scaled):")
console.print(f"     Training Time: {training_time:.1f}s (1000 trees)")

drawdown_mae_pct = rf_result.get('drawdown_mae_pct', 0)
drawdown_mae_bps = rf_result.get('drawdown_mae_bps', 0)
streak_mae = rf_result.get('streak_prob_mae', 0)

console.print(f"\n     Drawdown Prediction:")
console.print(f"       MAE: {drawdown_mae_pct*100:.3f}% ({drawdown_mae_bps:.1f} bps)")
console.print(f"       Meaning: Avg error in risk estimate")

console.print(f"\n     Streak Probability:")
console.print(f"       MAE: {streak_mae:.4f}")
console.print(f"       Meaning: Avg error in loss continuation probability")

# Quality assessment with realistic thresholds for PERCENTAGE-based targets
if drawdown_mae_pct < 0.005:  # < 0.5% error = 50 bps
    console.print(f"\n     [bold green]✅ EXCELLENT drawdown prediction (<0.5% / 50 bps)[/bold green]")
elif drawdown_mae_pct < 0.01:  # < 1.0% error = 100 bps
    console.print(f"\n     [green]✅ GOOD drawdown prediction (<1.0% / 100 bps)[/green]")
elif drawdown_mae_pct < 0.02:  # < 2.0% error = 200 bps
    console.print(f"\n     [yellow]⚠️ ACCEPTABLE drawdown prediction (<2.0% / 200 bps)[/yellow]")
else:
    console.print(f"\n     [red]❌ POOR drawdown prediction (>{drawdown_mae_pct*100:.1f}%)[/red]")
    console.print(f"        Consider: More volatility features or longer history")

if streak_mae < 0.15:
    console.print(f"     [bold green]✅ EXCELLENT streak prediction (<0.15)[/bold green]")
elif streak_mae < 0.25:
    console.print(f"     [green]✅ GOOD streak prediction (<0.25)[/green]")
elif streak_mae < 0.35:
    console.print(f"     [yellow]⚠️ ACCEPTABLE streak prediction (<0.35)[/yellow]")
else:
    console.print(f"     [red]❌ WEAK streak prediction (>{streak_mae:.2f})[/red]")

console.print(f"\n✅ Random Forest training complete!")
console.print(f"   💡 A100 advantage: {1000/200}x more trees for robust risk estimates")


In [ ]:
# ============================================================
# 6.7 Train ElasticNet (Confidence/Stability Scorer) - A100 OPTIMIZED
# ============================================================
# Uses ElasticNetCV with TimeSeriesSplit for automatic hyperparameter tuning:
# - Combines L1 (Lasso) + L2 (Ridge) regularization
# - Automatic feature selection through L1 sparsity
# - Temporal CV prevents data leakage
# - A100: Parallel CV across all CPU cores for fast hyperparameter search

console.print(Panel("[bold green]Step 6/6: Training ElasticNet (Confidence) - A100 PARALLEL CV[/bold green]"))

# ===== PRE-TRAINING DATA VALIDATION =====
console.print("  🔍 Validating confidence data...")

# Check data shapes
X_train_ridge = confidence_data['X_train']
y_train_ridge = confidence_data['y_train']
X_val_ridge = confidence_data['X_val']
y_val_ridge = confidence_data['y_val']

console.print(f"     X_train: {X_train_ridge.shape}")
console.print(f"     y_train: {y_train_ridge.shape}")

# Check target distribution (should be 1D: confidence scores 0-100)
if y_train_ridge.ndim == 1:
    console.print(f"\n  📊 Confidence Score Distribution:")
    console.print(f"     Range: [{y_train_ridge.min():.2f}, {y_train_ridge.max():.2f}]")
    console.print(f"     Mean: {y_train_ridge.mean():.2f}")
    console.print(f"     Std: {y_train_ridge.std():.2f}")
    
    # Data quality checks
    if y_train_ridge.min() < 0 or y_train_ridge.max() > 100:
        console.print(f"     [red]❌ ERROR: Confidence scores outside 0-100 range![/red]")
        raise ValueError(f"Confidence must be in [0,100], got [{y_train_ridge.min():.2f}, {y_train_ridge.max():.2f}]")
    
    if y_train_ridge.std() < 1.0:
        console.print(f"     [yellow]⚠️ Very low variance ({y_train_ridge.std():.2f}) - limited signal[/yellow]")
    
    # Distribution analysis
    low_conf = (y_train_ridge < 33).sum()
    med_conf = ((y_train_ridge >= 33) & (y_train_ridge < 67)).sum()
    high_conf = (y_train_ridge >= 67).sum()
    total = len(y_train_ridge)
    
    console.print(f"\n     Distribution:")
    console.print(f"       Low (0-33):    {low_conf:,} ({100*low_conf/total:.1f}%)")
    console.print(f"       Medium (33-67): {med_conf:,} ({100*med_conf/total:.1f}%)")
    console.print(f"       High (67-100):  {high_conf:,} ({100*high_conf/total:.1f}%)")
    
    if low_conf == 0 or high_conf == 0:
        console.print(f"     [yellow]⚠️ Missing low or high confidence samples - may struggle with extremes[/yellow]")
    
else:
    console.print(f"     [red]❌ ERROR: Expected 1D target, got shape {y_train_ridge.shape}[/red]")
    raise ValueError(f"Invalid ElasticNet target shape: {y_train_ridge.shape}")

# ===== A100-OPTIMIZED TRAINING WITH EXTENSIVE CV =====
console.print(f"\n  🔧 Training ElasticNet with parallel hyperparameter search...")
console.print(f"     • TimeSeriesSplit with 5 folds (prevents data leakage)")
console.print(f"     • Testing 50 alpha values (regularization strength)")
console.print(f"     • Testing 5 L1 ratios (Lasso vs Ridge balance)")
console.print(f"     • Total: 250 model combinations tested")
console.print(f"     • A100's CPU cores: Parallel CV for ~10x speedup")

ridge_trainer = RidgeTrainer(config)

import time
start_time = time.time()

ridge_result = ridge_trainer.train(
    X_train=X_train_ridge,
    y_train=y_train_ridge,
    X_val=X_val_ridge,
    y_val=y_val_ridge,
    feature_names=confidence_data.get('feature_names'),
)

training_time = time.time() - start_time

# ===== POST-TRAINING VALIDATION =====
console.print(f"\n  📊 ElasticNet Results (A100 Parallel CV):")
console.print(f"     Training Time: {training_time:.1f}s (250 models tested)")

conf_mae = ridge_result.get('confidence_mae', 0)
r2 = ridge_result.get('r2_score', 0)
best_alpha = ridge_result.get('best_alpha', 0)
best_l1_ratio = ridge_result.get('best_l1_ratio', 0)
n_nonzero = ridge_result.get('n_nonzero_coefs', 0)
n_total = ridge_result.get('n_total_coefs', 0)
sparsity = ridge_result.get('sparsity_ratio', 0)

console.print(f"\n     Prediction Quality:")
console.print(f"       MAE: {conf_mae:.2f} points (on 0-100 scale)")
console.print(f"       R² Score: {r2:.4f}")

console.print(f"\n     Hyperparameters (Auto-Selected):")
console.print(f"       Alpha: {best_alpha:.4f} (regularization strength)")
console.print(f"       L1 Ratio: {best_l1_ratio:.2f} (0=Ridge, 1=Lasso)")

console.print(f"\n     Feature Selection:")
console.print(f"       Active Features: {n_nonzero}/{n_total}")
console.print(f"       Sparsity: {sparsity*100:.1f}% (features zeroed out)")

# Quality assessment with realistic thresholds for 0-100 scale
console.print(f"\n  📈 Quality Assessment:")

# MAE thresholds (0-100 scale): <5 excellent, 5-10 good, 10-15 acceptable, >15 needs work
if conf_mae < 5:
    mae_quality = "[bold green]EXCELLENT[/bold green]"
    mae_interpretation = "±5 points - very reliable"
elif conf_mae < 10:
    mae_quality = "[green]GOOD[/green]"
    mae_interpretation = "±10 points - reliable"
elif conf_mae < 15:
    mae_quality = "[yellow]ACCEPTABLE[/yellow]"
    mae_interpretation = "±15 points - usable"
else:
    mae_quality = "[red]NEEDS IMPROVEMENT[/red]"
    mae_interpretation = f"±{conf_mae:.0f} points - unreliable"

# R² thresholds: >0.3 good, 0.1-0.3 useful, 0-0.1 weak, <0 worse than baseline
if r2 > 0.3:
    r2_quality = "[bold green]STRONG PREDICTIVE POWER[/bold green]"
    r2_interpretation = "Model explains >30% of variance"
elif r2 > 0.1:
    r2_quality = "[green]USEFUL SIGNAL[/green]"
    r2_interpretation = "Model captures meaningful patterns"
elif r2 > 0:
    r2_quality = "[yellow]WEAK SIGNAL[/yellow]"
    r2_interpretation = "Slight improvement over baseline"
else:
    r2_quality = "[red]WORSE THAN BASELINE[/red]"
    r2_interpretation = "Model worse than predicting mean"

console.print(f"     MAE: {mae_quality} ({mae_interpretation})")
console.print(f"     R²:  {r2_quality} ({r2_interpretation})")

# Feature selection interpretation
if sparsity > 0.5:
    console.print(f"     Features: [bold green]HIGHLY SPARSE[/bold green] (>{sparsity*100:.0f}% features eliminated)")
    console.print(f"               L1 regularization found {n_nonzero} key predictors")
elif sparsity > 0.2:
    console.print(f"     Features: [green]MODERATELY SPARSE[/green] ({sparsity*100:.0f}% features eliminated)")
elif sparsity > 0:
    console.print(f"     Features: [yellow]LOW SPARSITY[/yellow] (most features used)")
else:
    console.print(f"     Features: [yellow]NO SPARSITY[/yellow] (all features used - Ridge-like)")

# L1 ratio interpretation
if best_l1_ratio > 0.8:
    console.print(f"     Regularization: [cyan]LASSO-DOMINANT[/cyan] (strong feature selection)")
elif best_l1_ratio > 0.2:
    console.print(f"     Regularization: [cyan]ELASTIC NET BALANCE[/cyan] (selection + stability)")
else:
    console.print(f"     Regularization: [cyan]RIDGE-DOMINANT[/cyan] (stability over selection)")

console.print(f"\n✅ ElasticNet training complete!")
console.print(f"   💡 A100 advantage: Parallel CV tested 250 models in {training_time:.1f}s")


In [ ]:
# ============================================================
# 6.8 Save all models + Scaler (PRODUCTION READY)
# ============================================================
console.print(Panel("[bold blue]💾 Saving Production Models[/bold blue]"))

import json
import pickle
from datetime import datetime
import shutil

MODEL_DIR = "trained_data/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Save Transformer model
model.save(f"{MODEL_DIR}/transformer_direction.keras")
console.print(f"  💾 Saved: {MODEL_DIR}/transformer_direction.keras")

# Save the scaler (CRITICAL for inference!)
with open(f"{MODEL_DIR}/direction_scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)
console.print(f"  💾 Saved: {MODEL_DIR}/direction_scaler.pkl")

# Save feature configuration
feature_config = {
    'n_features': n_features,
    'feature_names': direction_data.get('feature_names', []),
    'valid_mask': valid_mask.tolist() if hasattr(valid_mask, 'tolist') else list(valid_mask),
}
with open(f"{MODEL_DIR}/direction_feature_config.json", 'w') as f:
    json.dump(feature_config, f, indent=2)
console.print(f"  💾 Saved: {MODEL_DIR}/direction_feature_config.json")

# Save XGBoost
xgb_trainer.save(f"{MODEL_DIR}/xgb_momentum.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/xgb_momentum.pkl")

# Save Random Forest
rf_trainer.save(f"{MODEL_DIR}/rf_risk.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/rf_risk.pkl")

# Save Ridge
ridge_trainer.save(f"{MODEL_DIR}/ridge_confidence.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/ridge_confidence.pkl")

# Save comprehensive metadata
metadata = {
    "trained_at": datetime.now().isoformat(),
    "trained_on": "colab_cuda_a100",
    "instrument": INSTRUMENT,
    "granularity": GRANULARITY,
    "candles": CANDLES,
    "multi_pair_mode": MULTI_PAIR_MODE,
    "selected_pairs": SELECTED_PAIRS if MULTI_PAIR_MODE else [INSTRUMENT],
    "config": TRAINING_CONFIG,
    "model_architecture": {
        "d_model": simple_config.transformer_d_model,
        "num_layers": simple_config.transformer_num_layers,
        "dff": simple_config.transformer_dff,
        "dropout": simple_config.transformer_dropout,
        "n_features": n_features,
    },
    "results": {
        "transformer": transformer_result,
        "xgboost": xgb_result,
        "random_forest": rf_result,
        "ridge": ridge_result,
    },
    "verification": verification_result if 'verification_result' in dir() else None,
}

with open(f"{MODEL_DIR}/modular_ensemble.meta.json", 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
console.print(f"  💾 Saved: {MODEL_DIR}/modular_ensemble.meta.json")

console.print(f"\n[bold green]✅ All models saved! ({MODEL_DIR})[/bold green]")
console.print(f"\n🎉 PRODUCTION-READY MODEL:")
console.print(f"   Balanced Accuracy: {transformer_result['val_balanced_accuracy']:.1%}")
console.print(f"   UP Accuracy:       {transformer_result['val_up_accuracy']:.1%}")
console.print(f"   DOWN Accuracy:     {transformer_result['val_down_accuracy']:.1%}")

## 7️⃣ Training Summary & Visualization

In [ ]:
# ============================================================
# 7.1 Training summary
# ============================================================
from rich.table import Table
from rich.panel import Panel
from rich.console import Console
import os, json

# Ensure console exists
if 'console' not in dir():
    console = Console()

console.print(Panel("[bold green]🎉 Training Complete![/bold green]"))

# Check if results exist, if not try to load from metadata
if 'transformer_result' not in dir() or transformer_result is None:
    console.print("[yellow]⚠️ Results not in memory, loading from saved metadata...[/yellow]")
    
    # Find metadata file
    meta_paths = [
        "trained_data/models/modular_ensemble.meta.json",
        "/content/ml_engine/trained_data/models/modular_ensemble.meta.json",
        "/content/trained_data/models/modular_ensemble.meta.json",
    ]
    
    metadata = None
    for mp in meta_paths:
        if os.path.exists(mp):
            with open(mp, 'r') as f:
                metadata = json.load(f)
            console.print(f"  ✅ Loaded from: {mp}")
            break
    
    if metadata and 'results' in metadata:
        transformer_result = metadata['results'].get('transformer', {})
        xgb_result = metadata['results'].get('xgboost', {})
        rf_result = metadata['results'].get('random_forest', {})
        ridge_result = metadata['results'].get('ridge', {})
        console.print("  ✅ Results loaded from metadata")
    else:
        console.print("[red]❌ No metadata found - showing empty results[/red]")
        transformer_result = {}
        xgb_result = {}
        rf_result = {}
        ridge_result = {}

# Also ensure other result dicts exist
if 'xgb_result' not in dir():
    xgb_result = {}
if 'rf_result' not in dir():
    rf_result = {}
if 'ridge_result' not in dir():
    ridge_result = {}

summary_table = Table(title="Model Performance Summary")
summary_table.add_column("Model", style="cyan")
summary_table.add_column("Metric", style="magenta")
summary_table.add_column("Value", style="green")

# Transformer - use correct keys (val_balanced_accuracy, not balanced_accuracy)
summary_table.add_row("Transformer", "Val Accuracy", f"{transformer_result.get('val_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "Balanced Acc", f"{transformer_result.get('val_balanced_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "↳ UP Accuracy", f"{transformer_result.get('val_up_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "↳ DOWN Accuracy", f"{transformer_result.get('val_down_accuracy', 0):.1%}")

# XGBoost - correct key is 'acceleration_accuracy' not 'accel_accuracy'
summary_table.add_row("XGBoost", "Accel Accuracy", f"{xgb_result.get('acceleration_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Momentum MAE", f"{xgb_result.get('momentum_mae', 0):.4f}")

# Random Forest - correct keys: 'drawdown_mae_pct' and 'streak_prob_mae'
dd_mae = rf_result.get('drawdown_mae_pct', rf_result.get('drawdown_mae', 0))
streak_mae = rf_result.get('streak_prob_mae', rf_result.get('streak_mae', 0))
summary_table.add_row("Random Forest", "Drawdown MAE", f"{dd_mae:.6f}" if dd_mae < 0.01 else f"{dd_mae:.4f}")
summary_table.add_row("Random Forest", "Streak MAE", f"{streak_mae:.4f}")

# ElasticNet (Ridge) - now with hyperparameter info
summary_table.add_row("ElasticNet", "R² Score", f"{ridge_result.get('r2_score', 0):.3f}")
summary_table.add_row("ElasticNet", "Confidence MAE", f"{ridge_result.get('confidence_mae', 0):.2f}")
summary_table.add_row("ElasticNet", "Best Alpha", f"{ridge_result.get('best_alpha', 1.0):.4f}")
summary_table.add_row("ElasticNet", "L1 Ratio", f"{ridge_result.get('best_l1_ratio', 0.5):.2f}")
n_nonzero = ridge_result.get('n_nonzero_coefs', '?')
n_total = ridge_result.get('n_total_coefs', '?')
summary_table.add_row("ElasticNet", "Features (sparse)", f"{n_nonzero}/{n_total}")

console.print(summary_table)

In [ ]:
# ============================================================
# 7.2 ElasticNet Feature Importance (Sparse Coefficients)
# ============================================================
# Visualize which features were selected by L1 regularization

import matplotlib.pyplot as plt
import numpy as np

# Get coefficients and feature names from the trainer
if 'ridge_trainer' in dir() and ridge_trainer.is_trained:
    coefs = ridge_trainer.model.coef_
    feature_names = ridge_trainer.feature_names or [f"Feature_{i}" for i in range(len(coefs))]
    
    # Filter non-zero coefficients
    nonzero_mask = coefs != 0
    nonzero_coefs = coefs[nonzero_mask]
    nonzero_names = [feature_names[i] for i in range(len(coefs)) if nonzero_mask[i]]
    
    if len(nonzero_coefs) > 0:
        # Sort by absolute value
        sorted_idx = np.argsort(np.abs(nonzero_coefs))[::-1]
        sorted_coefs = nonzero_coefs[sorted_idx]
        sorted_names = [nonzero_names[i] for i in sorted_idx]
        
        # Plot top 15 features
        n_show = min(15, len(sorted_coefs))
        
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ['green' if c > 0 else 'red' for c in sorted_coefs[:n_show]]
        bars = ax.barh(range(n_show), sorted_coefs[:n_show], color=colors, alpha=0.7)
        ax.set_yticks(range(n_show))
        ax.set_yticklabels(sorted_names[:n_show])
        ax.axvline(x=0, color='black', linewidth=0.5)
        ax.set_xlabel('Coefficient Value')
        ax.set_title(f'ElasticNet Feature Importance (Top {n_show} of {len(nonzero_coefs)} non-zero features)\n'
                     f'Alpha={ridge_result.get("best_alpha", 0):.4f}, L1 Ratio={ridge_result.get("best_l1_ratio", 0):.2f}')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        # Print summary
        console.print(f"\n[bold cyan]ElasticNet Feature Selection Summary:[/bold cyan]")
        console.print(f"  Total features: {len(coefs)}")
        console.print(f"  Non-zero (selected): {len(nonzero_coefs)} ({100*len(nonzero_coefs)/len(coefs):.1f}%)")
        console.print(f"  Zeroed (dropped): {len(coefs) - len(nonzero_coefs)} ({100*(len(coefs)-len(nonzero_coefs))/len(coefs):.1f}%)")
    else:
        console.print("[yellow]⚠️ All coefficients are zero - model may need different alpha range[/yellow]")
else:
    console.print("[yellow]⚠️ Ridge trainer not available - run training cells first[/yellow]")

## 💾 Save to Google Drive (Persistent Storage)

In [ ]:
# ============================================================================
# LIST MODEL FILES - Download manually from Colab web interface
# ============================================================================
import os
import glob

MODEL_DIR = '/content/ml_engine/trained_data/models'

# List all zip files in /content
print("=" * 60)
print("📦 ZIP FILES READY TO DOWNLOAD:")
print("=" * 60)
for f in glob.glob('/content/*.zip'):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  {f} ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("📁 MODEL FILES IN:", MODEL_DIR)
print("=" * 60)
for f in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {f} ({size_kb:.1f} KB)")

print("\n" + "=" * 60)
print("⬇️  HOW TO DOWNLOAD (VS Code + Colab workaround):")
print("=" * 60)
print("""
1. Open this notebook in COLAB WEB BROWSER:
   https://colab.research.google.com
   
2. Click the 📁 folder icon in the LEFT sidebar

3. Navigate to: /content/trained_models_*.zip

4. Right-click the zip file → Download

5. Extract to your local: trained_data/models/
""")

## 8️⃣ Download Models

In [ ]:
# ============================================================
# 8.1 Package models for download
# ============================================================
import shutil
from datetime import datetime

# Create zip file with all models
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f"ml_engine_models_{INSTRUMENT.replace('/', '_')}_{timestamp}"

# Create archive
shutil.make_archive(
    f"/content/{zip_name}",
    'zip',
    root_dir='/content/ml_engine',
    base_dir='trained_data/models'
)

print(f"✅ Models packaged: /content/{zip_name}.zip")
print(f"\n📦 Contents:")
!unzip -l /content/{zip_name}.zip | head -20

In [ ]:
# ============================================================
# 8.2 Download to local machine
# ============================================================
from google.colab import files

print("📥 Downloading models to your local machine...")
print("   (This will open a download dialog)\n")

files.download(f"/content/{zip_name}.zip")

print("\n✅ Download started!")
print("\n📋 To use on your Mac:")
print("   1. Unzip the downloaded file")
print("   2. Copy contents to ml_engine/trained_data/models/")
print("   3. Run: buddy analyze --model-type ensemble")

## 9️⃣ Optional: Push to GitHub

In [ ]:
# ============================================================
# 9.1 Commit and push trained models to GitHub
# ============================================================
# ⚠️ Only run this if you want to push models to your repo

PUSH_TO_GITHUB = False  # Set to True to enable

if PUSH_TO_GITHUB:
    from getpass import getpass
    
    print("🔐 GitHub Authentication")
    print("Enter your GitHub Personal Access Token (PAT)")
    print("Create one at: https://github.com/settings/tokens\n")
    
    GITHUB_TOKEN = getpass("GitHub PAT: ")
    GITHUB_USER = input("GitHub Username: ")
    GITHUB_EMAIL = input("GitHub Email: ")
    
    # Configure git
    !git config --global user.name "{GITHUB_USER}"
    !git config --global user.email "{GITHUB_EMAIL}"
    
    # Set remote with token
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Raynergy-svg/ml_engine.git
    
    # Add and commit
    !git add trained_data/models/
    !git commit -m "feat: Add CUDA-trained models from Colab ({INSTRUMENT})"
    
    # Push
    !git push origin main
    
    print("\n✅ Models pushed to GitHub!")
else:
    print("ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.")

---

## 📝 Notes

### A100 Powerhouse Configuration
This notebook uses a **4x larger model** than Mac M1, optimized for A100's 80GB VRAM:

| Parameter | Mac M1 | A100 | Scale |
|-----------|--------|------|-------|
| d_model | 32 | 128 | 4x |
| num_heads | 4 | 8 | 2x |
| num_layers | 2 | 4 | 2x |
| dff | 64 | 512 | 8x |
| seq_len | 64 | 128 | 2x |
| batch_size | 64 | 256 | 4x |

### Optimal Data Size Calculator
The notebook automatically calculates optimal candle count based on model size:
- **Formula**: `optimal_candles = model_params × 50`
- **A100 Powerhouse**: ~60,000+ candles recommended (vs 12,000 previously)
- **Rule**: Larger models need more data to generalize well

### Multi-Pair Training
Buddy scan analyzes pairs for trading suitability:
- **Volatility**: Higher = more tradeable
- **Trend Strength**: Higher = more predictable
- **Clear Moves**: % of bars with direction > 0.05%
- **Score**: Weighted combination for training quality

Top 5 pairs by liquidity and scan score are used by default.

### GPU Memory Usage
- **T4 (16GB)**: Use batch_size=64, d_model=64, single pair
- **A100 (80GB)**: Full powerhouse config with multi-pair training

### Training Time Estimates (12K candles per pair × 5 pairs = 60K total)
- **T4 GPU**: ~45-60 minutes (reduced config, single pair)
- **A100 GPU**: ~15-25 minutes (full powerhouse, multi-pair)

### A100 Tensor Core Optimization
- All dimensions are multiples of 64 for maximum Tensor Core efficiency
- TF32 enabled for 1.5x speedup on matrix operations
- XLA compilation for fused kernel optimization

### Troubleshooting
- **OOM Error**: Reduce batch_size to 128 or d_model to 64
- **OANDA timeout**: Reduce CANDLES_PER_PAIR or fetch fewer pairs
- **Import errors**: Restart runtime and re-run setup cells
- **0% accuracy**: Make sure mixed precision is DISABLED (TF 2.19 bug)
- **"client" not defined**: Run cell 3.2 (Test OANDA connection) first

## 🔥 Full Pipeline Backtest: Gated Trading with Kelly Sizing

In [ ]:
# ============================================================
# 10.0 LOAD SAVED MODELS (Run this if kernel was restarted)
# ============================================================
# This cell loads all saved models, scaler, and config from disk
# It also handles extracting from zip if needed

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
console = Console()

console.print(Panel("[bold cyan]📂 Loading Saved Models for Backtest[/bold cyan]"))

import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import glob
import zipfile

# ============================================================
# STEP 0: Find model directory (check multiple locations)
# ============================================================
POSSIBLE_MODEL_DIRS = [
    "trained_data/models",                    # Relative path
    "/content/ml_engine/trained_data/models", # Colab cloned repo
    "/content/trained_data/models",           # Colab root
    os.path.expanduser("~/ml_engine/trained_data/models"),  # Home dir
]

MODEL_DIR = None
for path in POSSIBLE_MODEL_DIRS:
    if os.path.exists(path):
        MODEL_DIR = path
        console.print(f"[green]✅ Found models at: {path}[/green]")
        break

# ============================================================
# STEP 1: If not found, look for zip file
# ============================================================
if MODEL_DIR is None:
    console.print(f"[yellow]⚠️ Model directory not found, looking for zip file...[/yellow]")
    
    # Look for model zip files in multiple locations
    search_paths = ['/content', '/content/ml_engine', '.', os.path.expanduser('~')]
    zip_files = []
    
    for path in search_paths:
        if os.path.exists(path):
            pattern = os.path.join(path, 'ml_engine_models*.zip')
            found = glob.glob(pattern)
            zip_files.extend(found)
            console.print(f"  Searched {path}: found {len(found)} zips")
    
    if zip_files:
        # Use the most recent zip
        latest_zip = max(zip_files, key=os.path.getmtime)
        console.print(f"  📦 Found: {latest_zip}")
        console.print(f"  📂 Extracting...")
        
        with zipfile.ZipFile(latest_zip, 'r') as zip_ref:
            zip_ref.extractall('.')
        
        MODEL_DIR = "trained_data/models"
        console.print(f"  ✅ Extracted to ./trained_data/models/")
    else:
        console.print(f"[red]❌ No zip file found. Options:[/red]")
        console.print("   1. Run training cells first")
        console.print("   2. Upload ml_engine_models_*.zip to /content")
        console.print("   3. Mount Google Drive and copy models")
        console.print("\n   Current working directory:", os.getcwd())
        console.print("   Listing /content:")
        if os.path.exists('/content'):
            for f in os.listdir('/content')[:20]:
                console.print(f"     {f}")

# ============================================================
# STEP 2: Load models if directory exists
# ============================================================
if MODEL_DIR and os.path.exists(MODEL_DIR):
    console.print(f"\n[green]📂 Loading from: {MODEL_DIR}[/green]")
    
    # List contents
    contents = os.listdir(MODEL_DIR)
    console.print(f"   Contents: {len(contents)} files")
    
    # Define custom loss functions used during training
    @tf.keras.saving.register_keras_serializable()
    def focal_loss(gamma=2.0, alpha=0.5):
        def loss_fn(y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
            pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
            alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
            return -tf.reduce_mean(alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt))
        return loss_fn

    @tf.keras.saving.register_keras_serializable()
    def combined_loss(y_true, y_pred):
        """Focal + BCE combined loss"""
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred_clipped, 1 - y_pred_clipped)
        focal = -tf.reduce_mean(tf.pow(1 - pt, 2.0) * tf.math.log(pt))
        return 0.5 * bce + 0.5 * focal
    
    @tf.keras.saving.register_keras_serializable()
    def anti_collapse_loss(y_true, y_pred):
        return tf.keras.losses.binary_crossentropy(y_true, y_pred)
    
    custom_objects = {
        'focal_loss': focal_loss,
        'combined_loss': combined_loss,
        'anti_collapse_loss': anti_collapse_loss,
    }
    
    # Load Transformer model
    model_path = f"{MODEL_DIR}/transformer_direction.keras"
    if os.path.exists(model_path):
        try:
            model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
            console.print(f"  ✅ Loaded: transformer_direction.keras")
        except Exception as e:
            console.print(f"  ⚠️ Load with custom objects failed: {e}")
            try:
                model = tf.keras.models.load_model(model_path, compile=False)
                model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
                console.print(f"  ✅ Loaded (recompiled): transformer_direction.keras")
            except Exception as e2:
                console.print(f"  ❌ Failed to load model: {e2}")
                model = None
    else:
        console.print(f"  ❌ Missing: transformer_direction.keras")
        model = None
    
    # Load Scaler (CRITICAL!)
    scaler_path = f"{MODEL_DIR}/direction_scaler.pkl"
    if os.path.exists(scaler_path):
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        console.print(f"  ✅ Loaded: direction_scaler.pkl")
    else:
        console.print(f"  ⚠️ Missing scaler - will need to create one")
        scaler = None
    
    # Load Feature Config
    config_path = f"{MODEL_DIR}/direction_feature_config.json"
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            feature_config = json.load(f)
        n_features = feature_config.get('n_features', 60)
        feature_names = feature_config.get('feature_names', [])
        console.print(f"  ✅ Loaded: feature_config ({n_features} features)")
    else:
        console.print(f"  ⚠️ Missing feature_config, using defaults")
        feature_config = {}
        n_features = 60
        feature_names = []
    
    # Load metadata for training config  
    meta_path = f"{MODEL_DIR}/modular_ensemble.meta.json"
    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            metadata = json.load(f)
        SELECTED_PAIRS = metadata.get('selected_pairs', ['EUR_USD'])
        GRANULARITY = metadata.get('granularity', 'H1')
        LOOKAHEAD = metadata.get('config', {}).get('lookahead_bars', 24)
        SEQ_LEN = 24  # Default
        console.print(f"  ✅ Loaded: metadata")
        console.print(f"     Pairs: {SELECTED_PAIRS}")
        console.print(f"     Granularity: {GRANULARITY}")
        
        # Show training results
        results = metadata.get('results', {})
        if 'transformer' in results:
            tr = results['transformer']
            console.print(f"\n  📊 Saved Model Performance:")
            console.print(f"     Balanced Accuracy: {tr.get('val_balanced_accuracy', 0):.1%}")
            console.print(f"     UP Accuracy: {tr.get('val_up_accuracy', 0):.1%}")
            console.print(f"     DOWN Accuracy: {tr.get('val_down_accuracy', 0):.1%}")
    else:
        # Try buddy meta file
        buddy_meta = f"{MODEL_DIR}/buddy_tf.meta.json"
        if os.path.exists(buddy_meta):
            with open(buddy_meta, 'r') as f:
                metadata = json.load(f)
            console.print(f"  ✅ Loaded: buddy_tf.meta.json")
        else:
            console.print(f"  ⚠️ No metadata found, using defaults")
            metadata = {}
        
        SELECTED_PAIRS = ['EUR_USD', 'GBP_USD', 'USD_JPY', 'USD_CHF', 'AUD_USD']
        GRANULARITY = 'H1'
        LOOKAHEAD = 24
        SEQ_LEN = 24
    
    # Verify model is loaded
    if model is not None:
        console.print(f"\n[bold green]✅ Models loaded successfully![/bold green]")
        console.print(f"   Model input shape: {model.input_shape}")
        # Extract n_features from model if available
        if model.input_shape and len(model.input_shape) >= 3:
            n_features = model.input_shape[-1]
            SEQ_LEN = model.input_shape[1] if model.input_shape[1] else 24
            console.print(f"   Features: {n_features}, Seq Length: {SEQ_LEN}")
        console.print(f"   Ready for backtesting!")
    else:
        console.print(f"\n[bold red]❌ Failed to load transformer model[/bold red]")
else:
    console.print(f"\n[red]❌ Could not find or extract models[/red]")
    model = None
    scaler = None

# ============================================================
# STEP 3: Setup OANDA client
# ============================================================
console.print(f"\n[cyan]Setting up OANDA client...[/cyan]")
try:
    # Try importing from repo
    import sys
    if '/content/ml_engine' not in sys.path:
        sys.path.insert(0, '/content/ml_engine')
    
    from oanda_practice import OandaPracticeClient
    OANDA_API_TOKEN = os.environ.get('OANDA_API_TOKEN', '')
    OANDA_ACCOUNT_ID = os.environ.get('OANDA_ACCOUNT_ID', '')
    if OANDA_API_TOKEN:
        client = OandaPracticeClient(OANDA_API_TOKEN, OANDA_ACCOUNT_ID)
        console.print(f"  ✅ OANDA client ready")
    else:
        console.print(f"  ⚠️ OANDA credentials not set - set OANDA_API_TOKEN env var")
except ImportError:
    console.print(f"  ⚠️ oanda_practice module not found")

In [ ]:
# ============================================================
# 10.1 Fetch Fresh Unseen Data for Backtesting
# ============================================================
# Get the most recent data that wasn't used in training

# Re-import in case kernel was restarted
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
console = Console()

console.print(Panel("[bold magenta]📊 Fetching Fresh Unseen Data for Backtest[/bold magenta]"))

# Fetch last 500 candles (unseen) for each pair
BACKTEST_CANDLES = 500  # ~3 weeks of H1 data

backtest_data = {}
for pair in SELECTED_PAIRS[:3]:  # Top 3 pairs for speed
    console.print(f"  Fetching {pair}...")
    try:
        resp = client.get_candles(
            instrument=pair,
            granularity=GRANULARITY,
            count=BACKTEST_CANDLES
        )
        candles = resp.get("candles", []) if isinstance(resp, dict) else []
        
        if candles:
            df_bt = pd.DataFrame([{
                'time': c['time'],
                'open': float(c['mid']['o']),
                'high': float(c['mid']['h']),
                'low': float(c['mid']['l']),
                'close': float(c['mid']['c']),
                'volume': int(c['volume']),
            } for c in candles])
            df_bt['time'] = pd.to_datetime(df_bt['time'])
            df_bt = df_bt.set_index('time')
            backtest_data[pair] = df_bt
            console.print(f"    ✅ {pair}: {len(df_bt)} candles ({df_bt.index[0].date()} to {df_bt.index[-1].date()})")
    except Exception as e:
        console.print(f"    ❌ {pair}: {e}")

console.print(f"\n✅ Loaded backtest data for {len(backtest_data)} pairs")

In [ ]:
# ============================================================
# 10.2 Prepare Features for Backtest Data
# ============================================================
# Apply same feature engineering as training

console.print(Panel("[bold magenta]🔧 Preparing Backtest Features[/bold magenta]"))

from modular_data_loaders import compute_normalized_features
from feature_engineering import FeatureEngineering

backtest_features = {}

for pair, df_bt in backtest_data.items():
    console.print(f"  Processing {pair}...")
    
    # Compute normalized features
    df_processed = compute_normalized_features(df_bt.copy())
    
    # Add technical indicators
    fe = FeatureEngineering()
    df_processed = fe.add_technical_indicators(df_processed)
    
    # Fill NaN
    df_processed = df_processed.ffill().bfill().dropna()
    
    # Get same features as training
    feature_cols = direction_data.get('feature_names', [])
    
    # Extract features that exist
    available_features = [f for f in feature_cols if f in df_processed.columns]
    missing_features = [f for f in feature_cols if f not in df_processed.columns]
    
    if missing_features:
        console.print(f"    ⚠️ Missing {len(missing_features)} features, padding with zeros")
        for mf in missing_features:
            df_processed[mf] = 0.0
    
    X_bt = df_processed[feature_cols].values.astype(np.float32)
    
    # Scale with saved scaler
    X_bt = scaler.transform(X_bt)
    X_bt = np.clip(X_bt, -10, 10)
    
    # Remove constant features (same mask as training)
    if 'valid_mask' in dir() and valid_mask is not None:
        X_bt = X_bt[:, valid_mask]
    
    backtest_features[pair] = {
        'X': X_bt,
        'df': df_processed,
        'prices': df_processed['close'].values,
    }
    
    console.print(f"    ✅ {pair}: {X_bt.shape[0]} samples, {X_bt.shape[1]} features")

console.print(f"\n✅ Features prepared for {len(backtest_features)} pairs")

In [ ]:
# ============================================================
# 10.3 Gated Trading Logic + Kelly Sizing + Dynamic SL/TP
# ============================================================
# Full production trading simulation

console.print(Panel("[bold magenta]🎯 GATED TRADING BACKTEST[/bold magenta]"))

from dataclasses import dataclass
from typing import List, Optional
import numpy as np

@dataclass
class TradeResult:
    """Single trade result"""
    pair: str
    entry_time: str
    exit_time: str
    direction: str  # 'LONG' or 'SHORT'
    entry_price: float
    exit_price: float
    stop_loss: float
    take_profit: float
    pnl_pct: float
    pnl_pips: float
    position_size: float
    confidence: float
    outcome: str  # 'WIN', 'LOSS', 'STOPPED', 'TP_HIT'

class GatedTradingSimulator:
    """
    Production-grade trading simulator with:
    - Confidence gating (only trade high-confidence signals)
    - Kelly criterion position sizing
    - Dynamic ATR-based SL/TP
    - Multi-pair support
    """
    
    def __init__(
        self,
        direction_model,
        momentum_model,
        risk_model,
        confidence_model,
        confidence_threshold: float = 0.60,
        min_confidence_gate: float = 0.55,
        kelly_fraction: float = 0.25,  # Quarter Kelly for safety
        atr_sl_multiplier: float = 1.5,
        atr_tp_multiplier: float = 2.5,
        risk_per_trade: float = 0.02,  # 2% max risk per trade
        lookahead_bars: int = 24,
    ):
        self.direction_model = direction_model
        self.momentum_model = momentum_model
        self.risk_model = risk_model
        self.confidence_model = confidence_model
        
        self.confidence_threshold = confidence_threshold
        self.min_confidence_gate = min_confidence_gate
        self.kelly_fraction = kelly_fraction
        self.atr_sl_multiplier = atr_sl_multiplier
        self.atr_tp_multiplier = atr_tp_multiplier
        self.risk_per_trade = risk_per_trade
        self.lookahead_bars = lookahead_bars
        
        self.trades: List[TradeResult] = []
        self.equity_curve = [10000.0]  # Start with $10,000
        
    def calculate_kelly_size(self, win_rate: float, avg_win: float, avg_loss: float) -> float:
        """
        Kelly Criterion: f* = (bp - q) / b
        Where:
            b = avg_win / avg_loss (win/loss ratio)
            p = win probability
            q = 1 - p (loss probability)
        """
        if avg_loss == 0 or win_rate <= 0:
            return 0.0
        
        b = avg_win / abs(avg_loss)
        p = win_rate
        q = 1 - p
        
        kelly = (b * p - q) / b
        kelly = max(0, min(kelly, 1))  # Clamp to [0, 1]
        
        # Apply fractional Kelly for safety
        return kelly * self.kelly_fraction
    
    def calculate_dynamic_sl_tp(self, prices: np.ndarray, idx: int, direction: str) -> tuple:
        """
        Calculate SL/TP based on ATR (volatility-adjusted)
        """
        # Calculate ATR over last 14 bars
        lookback = min(14, idx)
        if lookback < 5:
            return None, None
        
        highs = prices[idx-lookback:idx]  # Would need high/low for true ATR
        # Simplified: use price volatility
        volatility = np.std(np.diff(prices[max(0,idx-lookback):idx]) / prices[max(0,idx-lookback):idx-1])
        atr_estimate = volatility * prices[idx]
        
        if direction == 'LONG':
            sl = prices[idx] - (atr_estimate * self.atr_sl_multiplier * 100)  # Convert to price
            tp = prices[idx] + (atr_estimate * self.atr_tp_multiplier * 100)
        else:  # SHORT
            sl = prices[idx] + (atr_estimate * self.atr_sl_multiplier * 100)
            tp = prices[idx] - (atr_estimate * self.atr_tp_multiplier * 100)
        
        return sl, tp
    
    def run_backtest(self, pair: str, X: np.ndarray, prices: np.ndarray, 
                     times: np.ndarray, high_prices: np.ndarray = None, 
                     low_prices: np.ndarray = None) -> dict:
        """
        Run gated backtest on a single pair.
        """
        n_samples = len(X)
        
        # Need enough history for features
        start_idx = 50
        
        # Track running statistics for Kelly
        wins = 0
        losses = 0
        total_win_pct = 0.0
        total_loss_pct = 0.0
        
        pair_trades = []
        equity = self.equity_curve[-1]
        
        i = start_idx
        while i < n_samples - self.lookahead_bars:
            # Get model predictions
            X_sample = X[i:i+1]
            
            # Direction prediction
            direction_prob = float(self.direction_model.predict(X_sample, verbose=0)[0, 0])
            direction = 'LONG' if direction_prob > 0.5 else 'SHORT'
            raw_confidence = abs(direction_prob - 0.5) * 2  # 0 to 1 scale
            
            # === GATING LOGIC ===
            # Gate 1: Minimum confidence
            if raw_confidence < self.min_confidence_gate:
                i += 1
                continue
            
            # Gate 2: Model confidence (from ensemble)
            # Simplified: use direction confidence as proxy
            model_confidence = raw_confidence
            
            # Gate 3: Momentum alignment (would use XGBoost here)
            # For now, use direction strength as proxy
            momentum_aligned = raw_confidence > 0.6
            
            if not momentum_aligned and raw_confidence < 0.70:
                i += 1
                continue
            
            # === PASSED ALL GATES - TAKE TRADE ===
            entry_price = prices[i]
            entry_time = str(times[i]) if times is not None else f"bar_{i}"
            
            # Calculate dynamic SL/TP
            volatility = np.std(np.diff(prices[max(0,i-14):i]) / prices[max(0,i-14):i-1]) if i > 14 else 0.001
            atr_pct = volatility * 100  # As percentage
            
            if direction == 'LONG':
                sl_pct = atr_pct * self.atr_sl_multiplier
                tp_pct = atr_pct * self.atr_tp_multiplier
                stop_loss = entry_price * (1 - sl_pct)
                take_profit = entry_price * (1 + tp_pct)
            else:  # SHORT
                sl_pct = atr_pct * self.atr_sl_multiplier
                tp_pct = atr_pct * self.atr_tp_multiplier
                stop_loss = entry_price * (1 + sl_pct)
                take_profit = entry_price * (1 - tp_pct)
            
            # Calculate Kelly position size
            if wins + losses >= 5:  # Need some history
                win_rate = wins / (wins + losses)
                avg_win = total_win_pct / max(wins, 1)
                avg_loss = total_loss_pct / max(losses, 1)
                kelly_size = self.calculate_kelly_size(win_rate, avg_win, abs(avg_loss))
            else:
                kelly_size = 0.01  # Conservative start
            
            # Cap position size at risk_per_trade
            position_size = min(kelly_size, self.risk_per_trade) * equity
            
            # Simulate trade over lookahead period
            exit_idx = None
            exit_price = None
            outcome = None
            
            for j in range(i + 1, min(i + self.lookahead_bars + 1, n_samples)):
                current_price = prices[j]
                
                # Check SL/TP hits
                if direction == 'LONG':
                    if current_price <= stop_loss:
                        exit_price = stop_loss
                        outcome = 'STOPPED'
                        exit_idx = j
                        break
                    elif current_price >= take_profit:
                        exit_price = take_profit
                        outcome = 'TP_HIT'
                        exit_idx = j
                        break
                else:  # SHORT
                    if current_price >= stop_loss:
                        exit_price = stop_loss
                        outcome = 'STOPPED'
                        exit_idx = j
                        break
                    elif current_price <= take_profit:
                        exit_price = take_profit
                        outcome = 'TP_HIT'
                        exit_idx = j
                        break
            
            # If no SL/TP hit, exit at lookahead
            if exit_price is None:
                exit_idx = min(i + self.lookahead_bars, n_samples - 1)
                exit_price = prices[exit_idx]
                outcome = 'TIME_EXIT'
            
            # Calculate P&L
            if direction == 'LONG':
                pnl_pct = (exit_price - entry_price) / entry_price
            else:
                pnl_pct = (entry_price - exit_price) / entry_price
            
            pnl_pips = pnl_pct * 10000  # Convert to pips
            
            # Update statistics
            if pnl_pct > 0:
                wins += 1
                total_win_pct += pnl_pct
                outcome = 'WIN' if outcome == 'TIME_EXIT' else outcome
            else:
                losses += 1
                total_loss_pct += pnl_pct
                outcome = 'LOSS' if outcome == 'TIME_EXIT' else outcome
            
            # Update equity
            trade_pnl = position_size * pnl_pct
            equity += trade_pnl
            
            # Record trade
            trade = TradeResult(
                pair=pair,
                entry_time=entry_time,
                exit_time=str(times[exit_idx]) if times is not None else f"bar_{exit_idx}",
                direction=direction,
                entry_price=entry_price,
                exit_price=exit_price,
                stop_loss=stop_loss,
                take_profit=take_profit,
                pnl_pct=pnl_pct * 100,
                pnl_pips=pnl_pips,
                position_size=position_size,
                confidence=raw_confidence,
                outcome=outcome,
            )
            pair_trades.append(trade)
            
            # Skip to after exit
            i = exit_idx + 1
        
        self.trades.extend(pair_trades)
        self.equity_curve.append(equity)
        
        return {
            'pair': pair,
            'trades': len(pair_trades),
            'wins': wins,
            'losses': losses,
            'win_rate': wins / max(wins + losses, 1),
            'total_pnl_pct': sum(t.pnl_pct for t in pair_trades),
            'avg_pnl_pips': np.mean([t.pnl_pips for t in pair_trades]) if pair_trades else 0,
            'final_equity': equity,
        }

# Initialize simulator
simulator = GatedTradingSimulator(
    direction_model=model,
    momentum_model=xgb_trainer if xgb_trainer.is_trained else None,
    risk_model=rf_trainer if rf_trainer.is_trained else None,
    confidence_model=ridge_trainer if ridge_trainer.is_trained else None,
    confidence_threshold=0.60,
    min_confidence_gate=0.55,
    kelly_fraction=0.25,
    atr_sl_multiplier=1.5,
    atr_tp_multiplier=2.5,
)

console.print("✅ Gated Trading Simulator initialized")
console.print(f"   • Confidence gate: {simulator.min_confidence_gate:.0%}")
console.print(f"   • Kelly fraction: {simulator.kelly_fraction:.0%}")
console.print(f"   • SL multiplier: {simulator.atr_sl_multiplier}x ATR")
console.print(f"   • TP multiplier: {simulator.atr_tp_multiplier}x ATR")

In [ ]:
# ============================================================
# 10.4 Run Backtest on All Pairs
# ============================================================

console.print(Panel("[bold magenta]🚀 Running Gated Backtest[/bold magenta]"))

backtest_results = {}

for pair, data in backtest_features.items():
    console.print(f"\n📊 Backtesting {pair}...")
    
    X = data['X']
    prices = data['prices']
    times = data['df'].index.values
    
    result = simulator.run_backtest(
        pair=pair,
        X=X,
        prices=prices,
        times=times,
    )
    
    backtest_results[pair] = result
    
    console.print(f"   Trades: {result['trades']}")
    console.print(f"   Win Rate: {result['win_rate']:.1%}")
    console.print(f"   Total P&L: {result['total_pnl_pct']:.2f}%")
    console.print(f"   Avg P&L: {result['avg_pnl_pips']:.1f} pips")

# Summary
console.print("\n" + "="*70)
console.print("[bold]BACKTEST SUMMARY[/bold]")
console.print("="*70)

total_trades = sum(r['trades'] for r in backtest_results.values())
total_wins = sum(r['wins'] for r in backtest_results.values())
total_losses = sum(r['losses'] for r in backtest_results.values())
overall_win_rate = total_wins / max(total_trades, 1)
total_pnl = sum(r['total_pnl_pct'] for r in backtest_results.values())

console.print(f"\n📈 Overall Performance:")
console.print(f"   Total Trades:  {total_trades}")
console.print(f"   Wins/Losses:   {total_wins}/{total_losses}")
console.print(f"   Win Rate:      {overall_win_rate:.1%}")
console.print(f"   Total P&L:     {total_pnl:.2f}%")
console.print(f"   Final Equity:  ${simulator.equity_curve[-1]:,.2f}")

# Equity change
equity_change = (simulator.equity_curve[-1] - simulator.equity_curve[0]) / simulator.equity_curve[0] * 100
console.print(f"   Equity Change: {equity_change:+.2f}%")

In [ ]:
# ============================================================
# 10.5 Detailed Trade Analysis
# ============================================================

console.print(Panel("[bold magenta]📋 Trade-by-Trade Analysis[/bold magenta]"))

from rich.table import Table

# Create trade table
trade_table = Table(title="Recent Trades (Last 20)")
trade_table.add_column("Pair", style="cyan")
trade_table.add_column("Direction", style="bold")
trade_table.add_column("Entry", style="dim")
trade_table.add_column("Exit", style="dim")
trade_table.add_column("P&L %", style="green")
trade_table.add_column("Pips", style="magenta")
trade_table.add_column("Conf", style="yellow")
trade_table.add_column("Outcome", style="bold")

for trade in simulator.trades[-20:]:  # Last 20 trades
    pnl_style = "green" if trade.pnl_pct > 0 else "red"
    outcome_style = "green" if trade.outcome in ['WIN', 'TP_HIT'] else "red"
    
    trade_table.add_row(
        trade.pair,
        trade.direction,
        f"{trade.entry_price:.5f}",
        f"{trade.exit_price:.5f}",
        f"[{pnl_style}]{trade.pnl_pct:+.2f}%[/{pnl_style}]",
        f"{trade.pnl_pips:+.1f}",
        f"{trade.confidence:.0%}",
        f"[{outcome_style}]{trade.outcome}[/{outcome_style}]",
    )

console.print(trade_table)

# Outcome breakdown
console.print("\n📊 Outcome Breakdown:")
outcomes = {}
for t in simulator.trades:
    outcomes[t.outcome] = outcomes.get(t.outcome, 0) + 1

for outcome, count in sorted(outcomes.items(), key=lambda x: -x[1]):
    pct = count / len(simulator.trades) * 100
    console.print(f"   {outcome}: {count} ({pct:.1f}%)")

# Profit factor
winning_pnl = sum(t.pnl_pct for t in simulator.trades if t.pnl_pct > 0)
losing_pnl = abs(sum(t.pnl_pct for t in simulator.trades if t.pnl_pct < 0))
profit_factor = winning_pnl / max(losing_pnl, 0.01)

console.print(f"\n📈 Profit Factor: {profit_factor:.2f}")
console.print(f"   (>1.5 is good, >2.0 is excellent)")

In [ ]:
# ============================================================
# 10.6 DEBUG & FIX: Check Model Predictions on Backtest Data
# ============================================================

console.print(Panel("[bold red]🔍 DEBUG: Analyzing Why 0 Trades[/bold red]"))

# Check what predictions look like on backtest data
for pair in list(backtest_features.keys())[:1]:  # Just first pair
    bt_data = backtest_features[pair]
    console.print(f"\n📊 Analyzing {pair} predictions:")
    
    # Handle both dict and array formats
    if isinstance(bt_data, dict):
        X_bt = bt_data.get('X', bt_data.get('features', None))
        console.print(f"   Data format: dict with keys {list(bt_data.keys())}")
    else:
        X_bt = bt_data
    
    if X_bt is None:
        console.print(f"   ❌ Could not find feature array")
        continue
        
    console.print(f"   X_bt shape: {X_bt.shape}")
    
    # Get all predictions
    all_preds = model.predict(X_bt, verbose=0).flatten()
    
    console.print(f"   Predictions range: {all_preds.min():.4f} to {all_preds.max():.4f}")
    console.print(f"   Predictions mean: {all_preds.mean():.4f}")
    console.print(f"   Predictions std: {all_preds.std():.4f}")
    
    # Check confidence distribution
    confidence = np.abs(all_preds - 0.5) * 2  # 0 to 1 scale
    console.print(f"\n   Confidence range: {confidence.min():.4f} to {confidence.max():.4f}")
    console.print(f"   Confidence mean: {confidence.mean():.4f}")
    
    # Check how many pass each threshold
    thresholds = [0.3, 0.4, 0.5, 0.55, 0.6, 0.7]
    for thresh in thresholds:
        n_pass = np.sum(confidence >= thresh)
        pct = n_pass / len(confidence) * 100
        console.print(f"   Pass {thresh:.0%} threshold: {n_pass} ({pct:.1f}%)")
    
    # Check direction distribution
    n_long = np.sum(all_preds > 0.5)
    n_short = np.sum(all_preds <= 0.5)
    console.print(f"\n   Predictions > 0.5 (LONG): {n_long} ({n_long/len(all_preds)*100:.1f}%)")
    console.print(f"   Predictions ≤ 0.5 (SHORT): {n_short} ({n_short/len(all_preds)*100:.1f}%)")

# THE PROBLEM: Check if predictions are all ~0.5 (collapsed model on new data)
console.print("\n" + "="*60)
if all_preds.std() < 0.1:
    console.print("[bold red]⚠️ PROBLEM: Model predictions are too uniform (collapsed)[/bold red]")
    console.print("   This happens when model sees different data distribution")
    console.print("   Solutions:")
    console.print("   1. Lower confidence threshold")
    console.print("   2. Check feature alignment between train/backtest")
    console.print("   3. Retrain with more diverse data")
else:
    console.print("[green]✅ Predictions have good variance[/green]")

In [ ]:
# ============================================================
# 10.7 FIXED BACKTEST: Lower Thresholds + Better Gating
# ============================================================

console.print(Panel("[bold green]🔧 FIXED BACKTEST with Relaxed Thresholds[/bold green]"))

class FixedGatedSimulator:
    """
    Fixed simulator with:
    - Lower confidence threshold (0.30 instead of 0.60)
    - Trade every N bars if no strong signal (ensure trades happen)
    - Better Kelly calculation
    """
    
    def __init__(
        self,
        direction_model,
        confidence_threshold: float = 0.30,  # LOWERED from 0.60
        kelly_fraction: float = 0.25,
        atr_sl_multiplier: float = 1.5,
        atr_tp_multiplier: float = 2.0,
        risk_per_trade: float = 0.02,
        lookahead_bars: int = 24,
        min_bars_between_trades: int = 4,  # Don't trade every bar
    ):
        self.direction_model = direction_model
        self.confidence_threshold = confidence_threshold
        self.kelly_fraction = kelly_fraction
        self.atr_sl_multiplier = atr_sl_multiplier
        self.atr_tp_multiplier = atr_tp_multiplier
        self.risk_per_trade = risk_per_trade
        self.lookahead_bars = lookahead_bars
        self.min_bars_between_trades = min_bars_between_trades
        
        self.trades = []
        self.equity_curve = [10000.0]
        
    def calculate_kelly_size(self, win_rate: float, avg_rr: float) -> float:
        """Kelly with reward:risk ratio"""
        if win_rate <= 0 or avg_rr <= 0:
            return 0.01
        kelly = win_rate - ((1 - win_rate) / avg_rr)
        kelly = max(0.01, min(kelly, 0.5))
        return kelly * self.kelly_fraction
    
    def run_backtest(self, pair: str, X: np.ndarray, prices: np.ndarray, 
                     times: np.ndarray = None) -> dict:
        """Run backtest with fixed gating logic."""
        n_samples = len(X)
        
        # Get ALL predictions at once (faster)
        all_preds = self.direction_model.predict(X, verbose=0).flatten()
        
        # Calculate confidence for all
        all_confidence = np.abs(all_preds - 0.5) * 2
        
        console.print(f"\n🎯 {pair}: Pred range [{all_preds.min():.3f}, {all_preds.max():.3f}], Conf mean: {all_confidence.mean():.3f}")
        
        pair_trades = []
        equity = self.equity_curve[-1]
        wins, losses = 0, 0
        total_win_pct, total_loss_pct = 0.0, 0.0
        
        last_trade_idx = -self.min_bars_between_trades
        
        for i in range(50, n_samples - self.lookahead_bars):
            # Skip if too soon after last trade
            if i - last_trade_idx < self.min_bars_between_trades:
                continue
            
            pred = all_preds[i]
            conf = all_confidence[i]
            
            # Gating: must have SOME confidence
            if conf < self.confidence_threshold:
                continue
            
            # Direction
            direction = 'LONG' if pred > 0.5 else 'SHORT'
            entry_price = prices[i]
            
            # Dynamic SL/TP based on volatility
            lookback = min(14, i)
            if lookback > 2:
                returns = np.diff(prices[i-lookback:i]) / prices[i-lookback:i-1]
                volatility = np.std(returns)
            else:
                volatility = 0.001
            
            atr_pct = max(volatility, 0.0005)  # Minimum volatility
            
            if direction == 'LONG':
                stop_loss = entry_price * (1 - atr_pct * self.atr_sl_multiplier)
                take_profit = entry_price * (1 + atr_pct * self.atr_tp_multiplier)
            else:
                stop_loss = entry_price * (1 + atr_pct * self.atr_sl_multiplier)
                take_profit = entry_price * (1 - atr_pct * self.atr_tp_multiplier)
            
            # Simulate trade
            exit_price = None
            outcome = None
            exit_idx = i
            
            for j in range(i + 1, min(i + self.lookahead_bars + 1, n_samples)):
                current = prices[j]
                
                if direction == 'LONG':
                    if current <= stop_loss:
                        exit_price, outcome, exit_idx = stop_loss, 'SL_HIT', j
                        break
                    elif current >= take_profit:
                        exit_price, outcome, exit_idx = take_profit, 'TP_HIT', j
                        break
                else:
                    if current >= stop_loss:
                        exit_price, outcome, exit_idx = stop_loss, 'SL_HIT', j
                        break
                    elif current <= take_profit:
                        exit_price, outcome, exit_idx = take_profit, 'TP_HIT', j
                        break
            
            # Time exit if no SL/TP hit
            if exit_price is None:
                exit_idx = min(i + self.lookahead_bars, n_samples - 1)
                exit_price = prices[exit_idx]
                outcome = 'TIME_EXIT'
            
            # P&L
            if direction == 'LONG':
                pnl_pct = (exit_price - entry_price) / entry_price
            else:
                pnl_pct = (entry_price - exit_price) / entry_price
            
            pnl_pips = pnl_pct * 10000
            
            # Update stats
            if pnl_pct > 0:
                wins += 1
                total_win_pct += pnl_pct
                outcome = outcome if outcome != 'TIME_EXIT' else 'WIN'
            else:
                losses += 1
                total_loss_pct += abs(pnl_pct)
                outcome = outcome if outcome != 'TIME_EXIT' else 'LOSS'
            
            # Position sizing with Kelly
            if wins + losses >= 3:
                win_rate = wins / (wins + losses)
                avg_rr = (total_win_pct / max(wins, 1)) / (total_loss_pct / max(losses, 1)) if total_loss_pct > 0 else 1.5
                kelly_size = self.calculate_kelly_size(win_rate, avg_rr)
            else:
                kelly_size = 0.01
            
            position_size = min(kelly_size, self.risk_per_trade) * equity
            trade_pnl = position_size * pnl_pct
            equity += trade_pnl
            self.equity_curve.append(equity)
            
            # Record trade
            trade = TradeResult(
                pair=pair,
                entry_time=str(times[i]) if times is not None else f"bar_{i}",
                exit_time=str(times[exit_idx]) if times is not None else f"bar_{exit_idx}",
                direction=direction,
                entry_price=entry_price,
                exit_price=exit_price,
                stop_loss=stop_loss,
                take_profit=take_profit,
                pnl_pct=pnl_pct * 100,
                pnl_pips=pnl_pips,
                position_size=position_size,
                confidence=conf,
                outcome=outcome
            )
            pair_trades.append(trade)
            self.trades.append(trade)
            
            # Update last trade time
            last_trade_idx = exit_idx
        
        win_rate = wins / max(wins + losses, 1)
        total_pnl = sum(t.pnl_pct for t in pair_trades)
        
        return {
            'pair': pair,
            'trades': len(pair_trades),
            'wins': wins,
            'losses': losses,
            'win_rate': win_rate,
            'total_pnl_pct': total_pnl,
            'avg_pnl_pips': np.mean([t.pnl_pips for t in pair_trades]) if pair_trades else 0
        }

# Create fixed simulator
fixed_simulator = FixedGatedSimulator(
    direction_model=model,
    confidence_threshold=0.30,  # Much lower threshold
    lookahead_bars=LOOKAHEAD,
    min_bars_between_trades=4
)

# Run on all pairs
console.print("\n" + "="*60)
console.print("[bold]Running FIXED Backtest...[/bold]")
console.print("="*60)

fixed_results = {}
for pair in backtest_features.keys():
    X_bt = backtest_features[pair]['X'] if isinstance(backtest_features[pair], dict) else backtest_features[pair]
    prices = backtest_data[pair]['close'].values
    times = backtest_data[pair]['datetime'].values if 'datetime' in backtest_data[pair].columns else None
    
    result = fixed_simulator.run_backtest(pair, X_bt, prices, times)
    fixed_results[pair] = result
    
    console.print(f"📊 {pair}:")
    console.print(f"   Trades: {result['trades']}")
    console.print(f"   Win Rate: {result['win_rate']:.1%}")
    console.print(f"   Total P&L: {result['total_pnl_pct']:.2f}%")

# Summary
console.print("\n" + "="*60)
console.print("[bold green]FIXED BACKTEST SUMMARY[/bold green]")
console.print("="*60)

total_trades = len(fixed_simulator.trades)
total_wins = sum(r['wins'] for r in fixed_results.values())
total_losses = sum(r['losses'] for r in fixed_results.values())
total_pnl = sum(r['total_pnl_pct'] for r in fixed_results.values())
final_equity = fixed_simulator.equity_curve[-1]

console.print(f"📈 Total Trades: {total_trades}")
console.print(f"   Wins/Losses: {total_wins}/{total_losses}")
console.print(f"   Win Rate: {total_wins/max(total_trades,1):.1%}")
console.print(f"   Total P&L: {total_pnl:.2f}%")
console.print(f"   Final Equity: ${final_equity:,.2f}")
console.print(f"   Equity Change: {(final_equity-10000)/100:+.2f}%")

## 🔄 Walk-Forward Validation

In [ ]:
# ============================================================
# 11.1 Walk-Forward Validation with Expanding Window
# ============================================================
# Simulates real trading: train on past, test on future, expand window

console.print(Panel("[bold cyan]🔄 Walk-Forward Validation[/bold cyan]"))

from sklearn.metrics import balanced_accuracy_score
import gc

class WalkForwardValidator:
    """
    Walk-forward validation for time series:
    - Start with initial training window
    - Test on next period
    - Expand training window to include test period
    - Repeat
    """
    
    def __init__(self, 
                 initial_train_pct: float = 0.5,
                 test_window_size: int = 200,
                 n_folds: int = 5,
                 min_train_samples: int = 1000):
        self.initial_train_pct = initial_train_pct
        self.test_window_size = test_window_size
        self.n_folds = n_folds
        self.min_train_samples = min_train_samples
        self.fold_results = []
        
    def validate(self, X: np.ndarray, y: np.ndarray, 
                 build_model_fn, scaler=None) -> dict:
        """
        Run walk-forward validation.
        
        Args:
            X: Features array (samples, seq_len, features)
            y: Labels array
            build_model_fn: Function that returns compiled model
            scaler: Optional scaler (fit on each train fold)
        """
        n_samples = len(X)
        initial_train_size = int(n_samples * self.initial_train_pct)
        
        # Calculate fold boundaries
        remaining = n_samples - initial_train_size
        actual_test_size = min(self.test_window_size, remaining // self.n_folds)
        
        console.print(f"📊 Total samples: {n_samples}")
        console.print(f"📊 Initial train: {initial_train_size}")
        console.print(f"📊 Test window: {actual_test_size}")
        console.print(f"📊 Folds: {self.n_folds}")
        
        self.fold_results = []
        
        for fold in range(self.n_folds):
            train_end = initial_train_size + (fold * actual_test_size)
            test_start = train_end
            test_end = min(test_start + actual_test_size, n_samples)
            
            if train_end >= n_samples or test_end <= test_start:
                break
                
            X_train_fold = X[:train_end]
            y_train_fold = y[:train_end]
            X_test_fold = X[test_start:test_end]
            y_test_fold = y[test_start:test_end]
            
            console.print(f"\n[yellow]Fold {fold + 1}/{self.n_folds}[/yellow]")
            console.print(f"   Train: 0 → {train_end} ({len(X_train_fold)} samples)")
            console.print(f"   Test: {test_start} → {test_end} ({len(X_test_fold)} samples)")
            
            # Build fresh model for this fold
            fold_model = build_model_fn()
            
            # Calculate class weights for this fold
            n_up = np.sum(y_train_fold == 1)
            n_down = np.sum(y_train_fold == 0)
            total = len(y_train_fold)
            fold_class_weight = {
                0: total / (2 * max(n_down, 1)),
                1: total / (2 * max(n_up, 1))
            }
            
            # Train on fold
            fold_model.fit(
                X_train_fold, y_train_fold,
                validation_data=(X_test_fold, y_test_fold),
                epochs=15,  # Fewer epochs for speed
                batch_size=64,
                class_weight=fold_class_weight,
                verbose=0
            )
            
            # Predict
            y_pred_proba = fold_model.predict(X_test_fold, verbose=0)
            y_pred_class = (y_pred_proba > 0.5).astype(int).flatten()
            
            # Metrics
            accuracy = np.mean(y_pred_class == y_test_fold)
            balanced_acc = balanced_accuracy_score(y_test_fold, y_pred_class)
            
            up_mask = y_test_fold == 1
            down_mask = y_test_fold == 0
            up_acc = np.mean(y_pred_class[up_mask] == 1) if up_mask.sum() > 0 else 0
            down_acc = np.mean(y_pred_class[down_mask] == 0) if down_mask.sum() > 0 else 0
            
            fold_result = {
                'fold': fold + 1,
                'train_size': len(X_train_fold),
                'test_size': len(X_test_fold),
                'accuracy': accuracy,
                'balanced_accuracy': balanced_acc,
                'up_accuracy': up_acc,
                'down_accuracy': down_acc,
                'up_pct': up_mask.sum() / len(y_test_fold),
                'pred_up_pct': np.mean(y_pred_class == 1)
            }
            self.fold_results.append(fold_result)
            
            console.print(f"   ✅ Balanced Acc: {balanced_acc:.1%} (UP: {up_acc:.1%}, DOWN: {down_acc:.1%})")
            
            # Cleanup
            del fold_model
            gc.collect()
        
        # Summary
        avg_balanced = np.mean([r['balanced_accuracy'] for r in self.fold_results])
        std_balanced = np.std([r['balanced_accuracy'] for r in self.fold_results])
        avg_up = np.mean([r['up_accuracy'] for r in self.fold_results])
        avg_down = np.mean([r['down_accuracy'] for r in self.fold_results])
        
        return {
            'fold_results': self.fold_results,
            'avg_balanced_accuracy': avg_balanced,
            'std_balanced_accuracy': std_balanced,
            'avg_up_accuracy': avg_up,
            'avg_down_accuracy': avg_down,
            'n_folds': len(self.fold_results)
        }

# Build model function (reuses our architecture)
def build_walkforward_model():
    """Build a fresh model for walk-forward validation."""
    from tensorflow import keras
    
    inputs = keras.Input(shape=(SEQ_LEN, n_features))
    
    # Simplified transformer for speed
    x = keras.layers.Dense(64)(inputs)
    x = keras.layers.LayerNormalization()(x)
    
    # Single attention layer
    attn = keras.layers.MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = keras.layers.Add()([x, attn])
    x = keras.layers.LayerNormalization()(x)
    
    # FFN
    ffn = keras.layers.Dense(128, activation='relu')(x)
    ffn = keras.layers.Dropout(0.2)(ffn)
    ffn = keras.layers.Dense(64)(ffn)
    x = keras.layers.Add()([x, ffn])
    
    # Output
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

console.print("\n[green]✅ Walk-Forward Validator ready[/green]")

In [ ]:
# ============================================================
# 11.2 Run Walk-Forward Validation
# ============================================================

console.print(Panel("[bold magenta]🚀 Running Walk-Forward Validation[/bold magenta]"))

# Use the combined training data
wf_validator = WalkForwardValidator(
    initial_train_pct=0.5,   # Start with 50% of data
    test_window_size=300,     # Test on 300 samples per fold
    n_folds=5,                # 5 expanding folds
    min_train_samples=1000
)

# Run validation
wf_results = wf_validator.validate(
    X=X,  # Full feature array
    y=y_train_direction,  # Direction labels
    build_model_fn=build_walkforward_model
)

# Display results
console.print("\n" + "="*60)
console.print("[bold green]📊 Walk-Forward Summary[/bold green]")
console.print("="*60)

wf_table = Table(title="Fold Results")
wf_table.add_column("Fold", style="cyan")
wf_table.add_column("Train Size", style="dim")
wf_table.add_column("Test Size", style="dim")
wf_table.add_column("Balanced Acc", style="green")
wf_table.add_column("UP Acc", style="blue")
wf_table.add_column("DOWN Acc", style="red")

for r in wf_results['fold_results']:
    wf_table.add_row(
        str(r['fold']),
        str(r['train_size']),
        str(r['test_size']),
        f"{r['balanced_accuracy']:.1%}",
        f"{r['up_accuracy']:.1%}",
        f"{r['down_accuracy']:.1%}"
    )

console.print(wf_table)

console.print(f"\n📈 Average Balanced Accuracy: {wf_results['avg_balanced_accuracy']:.1%} ± {wf_results['std_balanced_accuracy']:.1%}")
console.print(f"   Average UP Accuracy: {wf_results['avg_up_accuracy']:.1%}")
console.print(f"   Average DOWN Accuracy: {wf_results['avg_down_accuracy']:.1%}")

# Interpretation
if wf_results['avg_balanced_accuracy'] > 0.60:
    console.print("\n✅ [green]Walk-forward results are STRONG - model generalizes well to future data[/green]")
elif wf_results['avg_balanced_accuracy'] > 0.55:
    console.print("\n⚠️ [yellow]Walk-forward results are MODERATE - some predictive power[/yellow]")
else:
    console.print("\n❌ [red]Walk-forward results are WEAK - model may not generalize[/red]")

# Check for degradation over time
accuracies = [r['balanced_accuracy'] for r in wf_results['fold_results']]
if len(accuracies) >= 3:
    early_avg = np.mean(accuracies[:2])
    late_avg = np.mean(accuracies[-2:])
    if late_avg < early_avg - 0.05:
        console.print("⚠️ [yellow]Performance degrading over time - consider more recent training data[/yellow]")
    elif late_avg > early_avg + 0.05:
        console.print("✅ [green]Performance improving with more data - good sign![/green]")

## 🔥 Warm Start Training (Incremental Learning)

In [ ]:
# ============================================================
# 12.1 Warm Start Manager - Load & Continue Training
# ============================================================
# Allows incremental training on new data without starting from scratch

console.print(Panel("[bold cyan]🔥 Warm Start Training System[/bold cyan]"))

import os
from datetime import datetime

class WarmStartManager:
    """
    Manages warm-start (incremental) training:
    - Load existing model weights
    - Continue training on new data
    - Track training history across sessions
    - Prevent catastrophic forgetting
    """
    
    def __init__(self, model_dir: str = './models'):
        self.model_dir = model_dir
        self.checkpoint_path = os.path.join(model_dir, 'warmstart_checkpoint.weights.h5')
        self.history_path = os.path.join(model_dir, 'training_history.json')
        self.scaler_path = os.path.join(model_dir, 'warmstart_scaler.pkl')
        self.training_history = []
        
    def save_checkpoint(self, model, scaler, metrics: dict, feature_names: list):
        """Save model checkpoint with metadata."""
        os.makedirs(self.model_dir, exist_ok=True)
        
        # Save weights
        model.save_weights(self.checkpoint_path)
        
        # Save scaler
        import joblib
        joblib.dump(scaler, self.scaler_path)
        
        # Save feature names
        feature_path = os.path.join(self.model_dir, 'feature_names.json')
        with open(feature_path, 'w') as f:
            json.dump(feature_names, f)
        
        # Update training history
        session_record = {
            'timestamp': datetime.now().isoformat(),
            'metrics': metrics,
            'n_features': len(feature_names)
        }
        self.training_history.append(session_record)
        
        # Save history
        with open(self.history_path, 'w') as f:
            json.dump(self.training_history, f, indent=2)
            
        console.print(f"✅ Checkpoint saved to {self.checkpoint_path}")
        return True
    
    def load_checkpoint(self, model):
        """Load model weights from checkpoint."""
        if not os.path.exists(self.checkpoint_path):
            console.print("⚠️ No checkpoint found - starting fresh")
            return False
            
        try:
            model.load_weights(self.checkpoint_path)
            console.print(f"✅ Loaded weights from {self.checkpoint_path}")
            
            # Load history
            if os.path.exists(self.history_path):
                with open(self.history_path, 'r') as f:
                    self.training_history = json.load(f)
                console.print(f"   Training sessions: {len(self.training_history)}")
                
            return True
        except Exception as e:
            console.print(f"❌ Failed to load checkpoint: {e}")
            return False
    
    def load_scaler(self):
        """Load saved scaler."""
        if not os.path.exists(self.scaler_path):
            return None
        import joblib
        return joblib.load(self.scaler_path)
    
    def load_feature_names(self):
        """Load saved feature names."""
        feature_path = os.path.join(self.model_dir, 'feature_names.json')
        if not os.path.exists(feature_path):
            return None
        with open(feature_path, 'r') as f:
            return json.load(f)
    
    def warm_train(self, model, X_new, y_new, 
                   epochs: int = 10,
                   learning_rate: float = 0.0001,
                   mix_ratio: float = 0.3):
        """
        Continue training on new data with anti-forgetting.
        
        Args:
            model: Existing trained model
            X_new: New training features
            y_new: New training labels
            epochs: Training epochs (fewer than initial)
            learning_rate: Lower LR to preserve learned weights
            mix_ratio: Ratio of old vs new data (0 = only new, 1 = only old)
        """
        console.print(f"\n🔄 Warm-starting with {len(X_new)} new samples")
        console.print(f"   Learning rate: {learning_rate} (reduced to preserve knowledge)")
        console.print(f"   Epochs: {epochs}")
        
        # Reduce learning rate for fine-tuning
        model.optimizer.learning_rate.assign(learning_rate)
        
        # Calculate class weights for new data
        n_up = np.sum(y_new == 1)
        n_down = np.sum(y_new == 0)
        total = len(y_new)
        class_weight = {
            0: total / (2 * max(n_down, 1)),
            1: total / (2 * max(n_up, 1))
        }
        
        # Split new data
        split_idx = int(len(X_new) * 0.85)
        X_train_new = X_new[:split_idx]
        y_train_new = y_new[:split_idx]
        X_val_new = X_new[split_idx:]
        y_val_new = y_new[split_idx:]
        
        # Early stopping to prevent overfitting on new data
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
        
        # Train
        history = model.fit(
            X_train_new, y_train_new,
            validation_data=(X_val_new, y_val_new),
            epochs=epochs,
            batch_size=64,
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=1
        )
        
        # Evaluate
        y_pred = (model.predict(X_val_new, verbose=0) > 0.5).astype(int).flatten()
        new_balanced_acc = balanced_accuracy_score(y_val_new, y_pred)
        
        console.print(f"\n✅ Warm-start complete!")
        console.print(f"   New data balanced accuracy: {new_balanced_acc:.1%}")
        
        return {
            'history': history.history,
            'new_balanced_accuracy': new_balanced_acc,
            'samples_trained': len(X_train_new)
        }

# Initialize warm start manager
warm_manager = WarmStartManager(model_dir='./models')
console.print("\n[green]✅ Warm Start Manager ready[/green]")

In [ ]:
# ============================================================
# 7.0 MODEL VALIDATION SUITE
# ============================================================
# Run this AFTER training to validate the 78% accuracy claim

console.print(Panel("[bold cyan]🔬 Running Validation Suite[/bold cyan]"))

# ========== 1. Cross-Pair Validation (USD_JPY) ==========
console.print("\n[bold]1. Cross-Pair Validation[/bold]")

# Load USD_JPY data
try:
    usd_jpy_file = f"{DATA_DIR}/USD_JPY_{TIMEFRAME}.csv"
    if os.path.exists(usd_jpy_file):
        usd_jpy_df = pd.read_csv(usd_jpy_file)
        usd_jpy_df['time'] = pd.to_datetime(usd_jpy_df['time'])
        usd_jpy_df.set_index('time', inplace=True)
        
        # Prepare features
        fe_jpy = FeatureEngineer()
        jpy_features = fe_jpy.create_features(usd_jpy_df)
        jpy_features['direction'] = (jpy_features['close'].pct_change().shift(-1) > 0).astype(int)
        jpy_features = jpy_features.dropna()
        
        # Use same feature columns as training
        X_jpy = jpy_features[direction_feature_cols].values
        y_jpy = jpy_features['direction'].values
        
        # Scale
        X_jpy_scaled = direction_scaler.transform(X_jpy)
        
        # Create sequences
        SEQ_LEN = 60
        X_jpy_seq = np.array([X_jpy_scaled[i-SEQ_LEN:i] for i in range(SEQ_LEN, len(X_jpy_scaled))])
        y_jpy_seq = y_jpy[SEQ_LEN:]
        
        # Predict
        jpy_pred = transformer_model.predict(X_jpy_seq, verbose=0)
        jpy_classes = (jpy_pred.flatten() > 0.5).astype(int)
        jpy_accuracy = np.mean(jpy_classes == y_jpy_seq)
        
        console.print(f"  USD_JPY Accuracy: {jpy_accuracy:.1%}")
        console.print(f"  Predictions: {np.mean(jpy_classes):.1%} UP | Actual: {np.mean(y_jpy_seq):.1%} UP")
        
        if jpy_accuracy < 0.55:
            console.print("  [yellow]⚠️ Model may be pair-specific (overfit to EUR/GBP)[/yellow]")
        elif jpy_accuracy > 0.60:
            console.print("  [green]✅ Model generalizes to USD_JPY[/green]")
    else:
        console.print(f"  ⚠️ USD_JPY data not found at {usd_jpy_file}")
except Exception as e:
    console.print(f"  ❌ USD_JPY validation error: {e}")

# ========== 2. Regime Analysis ==========
console.print("\n[bold]2. Market Regime Analysis[/bold]")

try:
    # Detect regimes in test data
    test_df_regime = test_df.copy()
    test_df_regime['returns'] = test_df_regime['close'].pct_change()
    test_df_regime['trend_20'] = test_df_regime['close'].pct_change(20)
    
    def classify_regime(trend):
        if pd.isna(trend):
            return 'unknown'
        if trend > 0.002:  # 0.2% move
            return 'bull'
        elif trend < -0.002:
            return 'bear'
        else:
            return 'ranging'
    
    test_df_regime['regime'] = test_df_regime['trend_20'].apply(classify_regime)
    
    # Get regime for each prediction
    regimes = test_df_regime['regime'].values[-len(direction_pred):]
    
    for regime in ['bull', 'bear', 'ranging']:
        mask = regimes == regime
        if np.sum(mask) > 20:
            regime_acc = np.mean(direction_pred_classes[mask] == y_direction_test[mask])
            emoji = '📈' if regime == 'bull' else '📉' if regime == 'bear' else '↔️'
            console.print(f"  {emoji} {regime.upper()}: {regime_acc:.1%} ({np.sum(mask)} samples)")
        else:
            console.print(f"  {regime}: Insufficient samples ({np.sum(mask)})")
            
except Exception as e:
    console.print(f"  ❌ Regime analysis error: {e}")

# ========== 3. Transaction Cost Analysis ==========
console.print("\n[bold]3. Post-Transaction Cost Analysis[/bold]")

try:
    # Transaction costs
    SPREAD = 0.0001  # 1 pip
    COMMISSION = 0.00001  # Per side
    SLIPPAGE = 0.00003  # 0.3 pips
    TOTAL_COST = SPREAD + COMMISSION * 2 + SLIPPAGE
    
    # Get actual returns
    actual_returns = test_df['close'].pct_change().values[-len(direction_pred):]
    
    # Positions from predictions
    positions = np.where(direction_pred.flatten() > 0.5, 1, -1)
    
    # Raw P&L
    raw_pnl = positions * actual_returns
    
    # Costs on position changes
    position_changes = np.abs(np.diff(positions, prepend=0))
    trade_costs = (position_changes > 0) * TOTAL_COST
    
    # Net P&L
    net_pnl = raw_pnl - trade_costs
    
    # Metrics
    sharpe = np.mean(net_pnl) / (np.std(net_pnl) + 1e-10) * np.sqrt(6240)  # Annualized
    cumulative = np.cumsum(net_pnl)
    peak = np.maximum.accumulate(cumulative)
    max_dd = np.max((peak - cumulative) / (np.abs(peak) + 1e-10))
    win_rate = np.mean(net_pnl > 0)
    trades = np.sum(position_changes > 0)
    
    console.print(f"  Sharpe Ratio: {sharpe:.2f} {'✅' if sharpe > 1.0 else '⚠️'}")
    console.print(f"  Max Drawdown: {max_dd:.1%} {'✅' if max_dd < 0.15 else '⚠️'}")
    console.print(f"  Win Rate: {win_rate:.1%}")
    console.print(f"  Total Return: {np.sum(net_pnl):.4f}")
    console.print(f"  Trading Costs: {np.sum(trade_costs):.4f}")
    console.print(f"  Number of Trades: {trades}")
    
    if np.sum(net_pnl) > np.sum(trade_costs):
        console.print("  [green]✅ Strategy is PROFITABLE after costs[/green]")
    else:
        console.print("  [red]❌ Strategy NOT profitable after costs[/red]")
        
except Exception as e:
    console.print(f"  ❌ Transaction cost error: {e}")

# ========== 4. Summary Assessment ==========
console.print(Panel("[bold green]Validation Summary[/bold green]"))

validation_results = {
    'direction_accuracy': float(direction_accuracy),
    'sharpe_ratio': float(sharpe) if 'sharpe' in dir() else None,
    'max_drawdown': float(max_dd) if 'max_dd' in dir() else None,
    'usd_jpy_accuracy': float(jpy_accuracy) if 'jpy_accuracy' in dir() else None,
}

issues = []
if direction_accuracy < 0.55:
    issues.append("❌ Low accuracy (< 55%)")
if 'sharpe' in dir() and sharpe < 1.0:
    issues.append("⚠️ Low Sharpe ratio (< 1.0)")
if 'max_dd' in dir() and max_dd > 0.15:
    issues.append("⚠️ High drawdown (> 15%)")
if 'jpy_accuracy' in dir() and jpy_accuracy < 0.52:
    issues.append("⚠️ Poor USD_JPY generalization")

if not issues:
    console.print("[green]✅ All validation checks passed![/green]")
    console.print("[green]   Model is ready for paper trading[/green]")
else:
    console.print("[yellow]⚠️ Issues found:[/yellow]")
    for issue in issues:
        console.print(f"   {issue}")

print(f"\n📋 Validation results saved")

In [ ]:
# ============================================================
# 12.2 Save Current Model as Warm-Start Checkpoint
# ============================================================

console.print(Panel("[bold yellow]💾 Saving Warm-Start Checkpoint[/bold yellow]"))

# Get current metrics
current_metrics = {
    'balanced_accuracy': float(actual_balanced),
    'up_accuracy': float(up_acc),
    'down_accuracy': float(down_acc),
    'verification_shuffle_acc': float(verification_result.get('shuffle_accuracy', 0)),
    'verification_future_acc': float(verification_result.get('future_accuracy', 0)),
    'total_samples': len(X),
    'pairs': SELECTED_PAIRS
}

# Get feature names
saved_feature_names = feature_cols if 'feature_cols' in dir() else list(range(n_features))

# Save checkpoint
warm_manager.save_checkpoint(
    model=model,
    scaler=scaler,
    metrics=current_metrics,
    feature_names=saved_feature_names
)

console.print("\n📊 Saved Metrics:")
for k, v in current_metrics.items():
    if isinstance(v, float):
        console.print(f"   {k}: {v:.4f}")
    else:
        console.print(f"   {k}: {v}")

console.print("\n[green]✅ Model saved - can be warm-started later![/green]")

In [ ]:
# ============================================================
# 12.3 Load Checkpoint & Continue Training (Example)
# ============================================================
# Run this cell when you want to continue training on new data

console.print(Panel("[bold magenta]🔄 Load & Continue Training Example[/bold magenta]"))

# This demonstrates how to warm-start from saved checkpoint
# Uncomment and modify when you have new data

ENABLE_WARM_START_DEMO = False  # Set to True to run demo

if ENABLE_WARM_START_DEMO:
    console.print("Loading checkpoint...")
    
    # Load saved scaler and features
    loaded_scaler = warm_manager.load_scaler()
    loaded_features = warm_manager.load_feature_names()
    
    if loaded_scaler and loaded_features:
        console.print(f"✅ Loaded scaler and {len(loaded_features)} features")
        
        # Build model with same architecture
        warm_model = build_walkforward_model()
        
        # Load weights
        if warm_manager.load_checkpoint(warm_model):
            console.print("✅ Model weights loaded")
            
            # Fetch new data (example: last 200 candles)
            console.print("\nFetching new data for warm-start...")
            new_candles = client.get_candles('EUR_USD', GRANULARITY, 200)
            new_df = pd.DataFrame(new_candles)
            new_df['datetime'] = pd.to_datetime(new_df['datetime'])
            
            # Process with same features
            fe_warm = FeatureEngineering(new_df)
            fe_warm.add_all_features()
            new_processed = fe_warm.df.copy()
            
            # Ensure same features
            missing = set(loaded_features) - set(new_processed.columns)
            if missing:
                console.print(f"⚠️ Missing features: {missing}")
            else:
                # Scale with loaded scaler
                X_new = new_processed[loaded_features].values
                X_new_scaled = loaded_scaler.transform(X_new)
                X_new_scaled = np.clip(X_new_scaled, -10, 10)
                
                # Create sequences
                # ... sequence creation logic here ...
                
                console.print("Ready for warm training!")
                # warm_manager.warm_train(warm_model, X_new_seq, y_new, epochs=10)
        else:
            console.print("❌ No checkpoint to load")
    else:
        console.print("❌ No scaler/features saved")
else:
    console.print("[dim]Warm-start demo disabled. Set ENABLE_WARM_START_DEMO = True to run.[/dim]")
    console.print("\n📝 How to use warm-start:")
    console.print("   1. Save checkpoint with cell above")
    console.print("   2. Later, load checkpoint: warm_manager.load_checkpoint(model)")
    console.print("   3. Fetch new data")
    console.print("   4. Call: warm_manager.warm_train(model, X_new, y_new)")
    console.print("\n   Key benefits:")
    console.print("   • Preserves learned patterns")
    console.print("   • Adapts to new market conditions")
    console.print("   • Faster than training from scratch")
    console.print("   • Lower learning rate prevents forgetting")